<div style="display:flex; align-items:center; justify-content:center; gap:22px; flex-wrap:wrap; width:100%; max-width:820px; margin:0 auto 16px; padding:10px 16px; box-sizing:border-box; background:#ffffff; border:1px solid #e5e7eb; border-radius:8px; box-shadow:0 1px 2px rgba(0,0,0,0.04);">
  <span style="display:flex; align-items:center; justify-content:center; flex:1.2 1 220px; min-width:170px; max-width:300px; height:74px;">
    <img src="assets/images/logos/nvidia-logo.png" alt="NVIDIA" style="display:block; max-width:100%; max-height:54px; width:auto; height:auto; object-fit:contain;">
  </span>
  <span style="display:flex; align-items:center; justify-content:center; flex:1.3 1 240px; min-width:190px; max-width:330px; height:74px;">
    <img src="assets/images/logos/udc.svg" alt="Universal Display Corporation" style="display:block; max-width:100%; max-height:42px; width:auto; height:auto; object-fit:contain;">
  </span>
  <span style="display:flex; align-items:center; justify-content:center; flex:1 1 150px; min-width:120px; max-width:200px; height:56px;">
    <img src="assets/images/logos/ovito_logo.png" alt="OVITO" style="display:block; max-width:100%; max-height:32px; width:auto; height:auto; object-fit:contain;">
  </span>
</div>

# Predicting Melting Points with <span style="color:#76b900; font-weight:600;">NVIDIA ALCHEMI</span>

<p style="margin-top:0; color:#555; font-size:0.95em;">
Inspiration for this tutorial is drawn from a collaboration between NVIDIA ALCHEMI and Universal Display Corporation to screen the melting points of candidate organic OLED molecules &mdash; a property that governs material selection, thermal stability, and the processing window of organic light-emitting diodes. OVITO is used throughout for structure inspection and rendering. The notebook below is a simplified melting-point tutorial inspired by that effort.
</p>

🔗 **ALCHEMI resources:** [ALCHEMI hub](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) · [Toolkit GitHub](https://github.com/NVIDIA/nvalchemi-toolkit) · [Toolkit-Ops GitHub](https://github.com/NVIDIA/nvalchemi-toolkit-ops)

📰 **ALCHEMI blogs:** [ALCHEMI discovery blog](https://developer.nvidia.com/blog/revolutionizing-ai-driven-material-discovery-using-nvidia-alchemi/) · [Toolkit intro blog](https://developer.nvidia.com/blog/building-custom-atomistic-simulation-workflows-for-chemistry-and-materials-science-with-nvidia-alchemi-toolkit/) · [Toolkit-Ops blog](https://developer.nvidia.com/blog/accelerating-ai-powered-chemistry-and-materials-science-simulations-with-nvidia-alchemi-toolkit-ops/)

![Melting-point screening with NVIDIA ALCHEMI Toolkit](assets/images/banner.jpg)

*This tutorial uses melting-point prediction as a worked example of a broader capability: GPU-native, batched molecular dynamics that lets a single researcher equilibrate a crystal, build a solid–liquid interface, and sweep temperatures in parallel — all from familiar Python structure tools.*

**What you will do**

1. Read an experimental naphthalene crystal and build a GPU-resident `Batch`.
2. Build a custom ALCHEMI model wrapper **from scratch** for the Orb-v3 (OMol) foundation model.
3. Equilibrate the crystal (FIRE → NVT → anisotropic NPT) and confirm it stays solid.
4. Assemble a solid–liquid coexistence cell, packing the liquid half with **Packmol**.
5. Sweep temperatures as one multi-graph batch and bracket the experimental $T_\mathrm{m}=353$ K.

## The melting point, the hard way and the direct way

The melting point $T_\mathrm{m}$ is one of the most basic — and most stubborn — properties to predict for a molecular crystal. It sets the processing window and thermal stability of pharmaceuticals, energetic materials, polymers, and the organic semiconductors used in displays. Yet a naive simulation gets it wrong: heating a perfect crystal in a single-phase NPT box **superheats** it, so it persists hundreds of kelvins above its true $T_\mathrm{m}$ before a liquid finally nucleates.

The **direct solid–liquid coexistence (SLC)** method sidesteps the nucleation barrier by placing a slab of solid against a slab of liquid in one periodic cell. The pre-formed interface removes the barrier, so the crystal grows below $T_\mathrm{m}$ and melts above it; at $T_\mathrm{m}$ the interface is stationary. Sweeping temperatures and watching which side advances brackets $T_\mathrm{m}$. We follow the SLC + rotational-order ($S_0$) + diffusion ($D$) screening procedure of Schmidt, Van der Spoel & Walz ([2023](https://doi.org/10.1021/acsphyschemau.2c00045)), applied here to naphthalene with the Orb-v3 machine-learned potential on the ALCHEMI Toolkit.

## Predicting melting is a long-timescale problem

Atomistic processes span an enormous range of timescales. A chemical bond vibrates every few femtoseconds; an elementary reaction crosses its barrier in a fraction of a picosecond; molecules reorient and diffuse over picoseconds to nanoseconds. Melting sits much further out — it is a **collective** process in which an entire solid–liquid interface must reorganize, and the front advances slowly enough that deciding which phase wins takes nanoseconds, orders of magnitude longer than the events most simulations are built around.

<div style="margin:18px auto 20px; max-width:920px; font-family:Inter, Arial, sans-serif; color:#1f2937;">
  <div style="text-align:center; font-weight:700; font-size:1.02em;">Physical timescales of atomistic processes</div>
  <div style="text-align:center; font-size:0.88em; color:#6b7280; margin:2px 0 12px;">each step down is roughly 10–1000× slower than the one above (log scale)</div>

  <div style="display:flex; align-items:center; gap:14px; margin:0 auto 7px; padding:10px 16px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; border-radius:8px;">
    <div style="flex:0 0 104px; text-align:right; font-family:ui-monospace, SFMono-Regular, Menlo, monospace; font-weight:700; color:#374151;">~10 fs</div>
    <div style="flex:1 1 auto;"><div style="font-weight:700;">Bond vibration</div><div style="font-size:0.9em; color:#4b5563;">C–H / C–C stretches; sets the MD timestep (Δt ≈ 0.5–1 fs)</div></div>
  </div>
  <div style="display:flex; align-items:center; gap:14px; margin:0 auto 7px; padding:10px 16px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; border-radius:8px;">
    <div style="flex:0 0 104px; text-align:right; font-family:ui-monospace, SFMono-Regular, Menlo, monospace; font-weight:700; color:#374151;">~0.1–1 ps</div>
    <div style="flex:1 1 auto;"><div style="font-weight:700;">Reactive event</div><div style="font-size:0.9em; color:#4b5563;">a single bond-breaking / forming barrier crossing</div></div>
  </div>
  <div style="display:flex; align-items:center; gap:14px; margin:0 auto 7px; padding:10px 16px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; border-radius:8px;">
    <div style="flex:0 0 104px; text-align:right; font-family:ui-monospace, SFMono-Regular, Menlo, monospace; font-weight:700; color:#374151;">~1–10 ps</div>
    <div style="flex:1 1 auto;"><div style="font-weight:700;">Molecular reorientation</div><div style="font-size:0.9em; color:#4b5563;">librations and rattling within the cage of neighbours</div></div>
  </div>

  <div style="display:flex; align-items:center; gap:10px; margin:10px auto;">
    <div style="flex:1; height:1px; background:#e5e7eb;"></div>
    <div style="font-size:0.8em; color:#9ca3af; white-space:nowrap;">↑ within reach of DFT <em>ab&nbsp;initio</em> MD &nbsp;·&nbsp; ↓ needs accelerated MLIP MD</div>
    <div style="flex:1; height:1px; background:#e5e7eb;"></div>
  </div>

  <div style="display:flex; align-items:center; gap:14px; margin:0 auto 7px; padding:10px 16px; box-sizing:border-box; border:1.5px solid #d1d5db; background:#ffffff; border-radius:8px;">
    <div style="flex:0 0 104px; text-align:right; font-family:ui-monospace, SFMono-Regular, Menlo, monospace; font-weight:700; color:#374151;">~0.1–10 ns</div>
    <div style="flex:1 1 auto;"><div style="font-weight:700;">Diffusion</div><div style="font-size:0.9em; color:#4b5563;">translational hopping; resolving a diffusion coefficient <em>D</em></div></div>
  </div>
  <div style="display:flex; align-items:center; gap:14px; margin:0 auto 7px; padding:11px 16px; box-sizing:border-box; border:2px solid #76b900; background:#edf7e2; border-radius:8px;">
    <div style="flex:0 0 104px; text-align:right; font-family:ui-monospace, SFMono-Regular, Menlo, monospace; font-weight:700; color:#18230f;">~1–100 ns</div>
    <div style="flex:1 1 auto;"><div style="font-weight:700; color:#18230f;">Melting / solid–liquid coexistence &nbsp;·&nbsp; <span style="color:#3d6b00;">this tutorial</span></div><div style="font-size:0.9em; color:#2f4711;">an interface advances or retreats until one phase consumes the other</div></div>
  </div>
  <div style="display:flex; align-items:center; gap:14px; margin:0 auto 0; padding:10px 16px; box-sizing:border-box; border:1.5px dashed #d1d5db; background:#fafafa; border-radius:8px; opacity:0.85;">
    <div style="flex:0 0 104px; text-align:right; font-family:ui-monospace, SFMono-Regular, Menlo, monospace; font-weight:700; color:#9ca3af;">~µs–s</div>
    <div style="flex:1 1 auto;"><div style="font-weight:700; color:#6b7280;">Nucleation &amp; crystallization</div><div style="font-size:0.9em; color:#9ca3af;">spontaneous birth of a new phase; usually needs enhanced sampling</div></div>
  </div>

  <div style="display:flex; justify-content:center; gap:10px; margin-top:14px; flex-wrap:wrap; font-size:0.82em;">
    <span style="padding:3px 11px; border:1px solid #d1d5db; background:#f9fafb; border-radius:999px; color:#4b5563;">DFT <em>ab initio</em> MD ≈ tens of ps</span>
    <span style="padding:3px 11px; border:1.5px solid #76b900; background:#edf7e2; border-radius:999px; color:#18230f; font-weight:600;">Accelerated MLIP MD on NVIDIA ALCHEMI ≈ ns and beyond</span>
  </div>
</div>

Born–Oppenheimer molecular dynamics with DFT forces is accurate enough for almost any chemistry, but its cost caps practical trajectories at roughly tens of picoseconds — far short of melting. Historically, the nanosecond-and-beyond timescales below the line were the territory of **empirical and semi-empirical force fields**: fast enough to get there, but limited in transferable accuracy for the diverse chemistries of molecular crystals. **Machine-learned interatomic potentials (MLIPs)** change the trade-off. Trained as surrogate models for *ab initio* DFT, they target near-DFT accuracy at force-field-like cost — pushing out the accuracy–speed Pareto front rather than forcing a choice between the two.

NVIDIA ALCHEMI turns that into practice. GPU-native, batched dynamics let a single researcher run the long solid–liquid coexistence trajectories — an entire temperature sweep at once — that brute-force ab-initio MD could never reach. That capability is exactly what makes the melting-point workflow in this notebook practical.

## Tools at a glance

Before the code starts, here is the role of each scientific tool in this workflow. The stack is intentionally interoperable: familiar structure tools in, the dynamics bottleneck accelerated on the GPU, and inspectable trajectories out.

| Tool | What it does here |
|---|---|
| **NVIDIA GPU / CUDA** | Supplies the accelerated execution layer that makes long, batched molecular dynamics — an entire temperature sweep at once — practical on one GPU. |
| **ALCHEMI Toolkit** | Connects atomistic structures to batched GPU dynamics: `AtomicData` / `Batch` containers, integrators (FIRE2, NVT-Langevin, anisotropic NPT), lifecycle hooks, and model wrappers. |
| **Orb-v3 (OMol)** | The machine-learned interatomic potential supplying energies, forces, and stress. We wrap it for the Toolkit **from scratch**, rather than using a shipped wrapper. |
| **D3(BJ)** | Grimme D3 dispersion with Becke–Johnson damping, composed with Orb-v3 to capture the van der Waals cohesion that holds a molecular crystal together. |
| **FIRE2** | *Fast Inertial Relaxation Engine* (v2): the Toolkit geometry optimizer used to relax structures before dynamics. |
| **NVT-Langevin / anisotropic NPT** | Thermostatted and barostatted MD ensembles; anisotropic NPT lets each cell axis respond independently — essential for a low-symmetry monoclinic crystal and a two-phase cell. |
| **Packmol** | Packs the liquid half of the coexistence cell, placing molecules around the fixed crystal slab without clashes ([Martínez et al., 2009](https://doi.org/10.1002/jcc.21224)). |
| **Solid–liquid coexistence (SLC)** | The melting-point workflow pattern: hold solid and liquid in one cell and watch which phase grows ([Schmidt et al., 2023](https://doi.org/10.1021/acsphyschemau.2c00045)). |
| **S₀ and D** | Rotational order parameter and translational diffusion coefficient — together they classify each phase as crystal, plastic, or liquid. |
| **ASE** | Reads the experimental naphthalene CIF, builds the supercell, and stores structures in Python. |
| **OVITO** | Scientific molecular-rendering platform used for the trajectory figures and animations. |

## Roadmap: equilibrate and check the model before melting anything

A melting point is only as trustworthy as the model underneath it, so the notebook earns that trust before it sweeps temperatures. It first builds the Orb-v3 wrapper and confirms the equilibrated crystal actually stays solid (rotational order $S_0 \to 1$, diffusion $D \approx 0$); only then does it assemble the two-phase cell and run the coexistence sweep.

The workflow proceeds in order:

- read the experimental naphthalene crystal and pack it into a Toolkit `Batch`;
- build a custom Orb-v3 (OMol) model wrapper from scratch and compose it with D3 dispersion;
- equilibrate the crystal — FIRE2 → NVT → anisotropic NPT — and confirm it is solid from its diagnostics;
- build the solid–liquid coexistence cell, packing the liquid half with Packmol;
- pre-equilibrate, then run anisotropic NPT across the temperature sweep as one multi-graph batch;
- read $S_0$ and $D$ per temperature and bracket the melting point against the experimental $T_\mathrm{m}=353$ K.

Two practical notes. The long NPT stages (≈50 ps warmup, ≈300 ps per coexistence temperature) are **not** meant to be run live in a tutorial setting; the notebook ships the cached trajectories, logs, and figures so the full story is visible offline, and a single `Config` switch chooses between replaying those saved results and recomputing the lightweight stages live. And because every diagnostic reads the same $S_0$ / $D$ phase classifier, the through-line from crystal to melt stays consistent across the whole notebook.

## Run configuration

Edit the cell below to choose how the notebook runs. Every section downstream reads the same `Config` object, so this is the only place you change anything.

- **`RESULT_SOURCE="saved"`** (default) replays the shipped results of the canonical run — cached logs, figures, animations, and thinned trajectories — so you can read the whole story without a GPU.
- **`RESULT_SOURCE="compute"`** runs the **lightweight** stages live: geometry relaxation (FIRE2) and a 1 ps thermalization for both the crystal and the coexistence cell, plus the Packmol build. The long equilibrations (≈50 ps warmup NPT, ≈300 ps coexistence NPT per temperature) are never run inline — their results always come from the cache.

The remaining toolkit knobs rarely change: `TOOLKIT_DEVICE="auto"` selects CUDA when available, `TOOLKIT_DTYPE` sets the working precision, and `TOOLKIT_COMPILE_MODEL` toggles `torch.compile` for the model (off by default; worthwhile only for long production reruns).

Everything below the toolkit knobs is the **canonical-run parameter set** — the exact configuration used to generate the saved results shipped with this notebook. It is shown for transparency and reproducibility; **leave it unchanged** unless you are regenerating the entire run yourself on a multi-GPU node, in which case the `"saved"` cache would no longer match.

`cfg.activate()` resolves the device, configures the torch runtime, prints a settings + package-version summary, and exports the ALL-CAPS names (`DEVICE`, `TEMPS`, `RUN_NAME`, …) that later cells reference.

In [ ]:
from helpers.config import Config

cfg = Config(
    # ── Run mode — safe to change ─────────────────────────────────────────
    RESULT_SOURCE="saved",         # "saved" replays the shipped cache; "compute" runs the light stages live
    TOOLKIT_DEVICE="auto",         # "auto", "cuda", or "cpu"
    TOOLKIT_DTYPE="float32",       # "float32" (recommended) or "float64"
    TOOLKIT_COMPILE_MODEL=False,   # torch.compile the MLIP; leave off for the tutorial
    # ── Canonical-run parameters — DO NOT ALTER ───────────────────────────
    # The exact settings behind the saved/cached results shipped with this
    # notebook. Editing any of these invalidates the "saved" cache; they are
    # only meaningful if you regenerate the full run yourself on a multi-GPU node.
    RUN_NAME="naphthalene_orbmol",
    TM_EXP=353.0,                  # K   — experimental naphthalene melting point
    T_WARMUP=200.0,                # K   — crystal warmup target
    T_MELT=500.0,                  # K   — Packmol liquid-half target
    TEMPS=(200.0, 300.0, 400.0, 500.0),  # K — coexistence temperature sweep
    SUPERCELL=(5, 5, 4),           # 200 molecules → 7200-atom coexistence stack
    DT=0.5,                        # fs  — MD timestep
    FRICTION=0.01,                 # fs⁻¹ — Langevin friction
    THERMOSTAT_TIME=100.0,         # fs  — Nosé–Hoover chain τ_T
    BAROSTAT_TIME=4000.0,          # fs  — MTK barostat τ_P
    FMAX=0.15,                     # eV/Å — FIRE2 force convergence
    WARMUP_NPT_PS=50.0,            # ps  — cached warmup NPT duration
    SLC_NPT_PS=300.0,              # ps  — cached coexistence NPT per temperature
)
cfg.activate()

### Imports

`cfg.activate()` already configured the torch runtime and exported the run parameters, so this cell only pulls in the **library-level** dependencies used across the notebook (ASE, PyTorch, NumPy, Matplotlib). Following the ALCHEMI Toolkit's compositional style, the interesting pieces — the custom model wrapper, the safety-hook chain, the Packmol builder — are **defined or imported inline** at their first use, so each pattern appears next to the code that explains it.

In [ ]:
# ─── Library-level imports ─────────────────────────────────────────────────
# cfg.activate() already resolved the device, set the torch backend flags, and
# exported the run parameters (DEVICE, DT, TEMPS, RUN_NAME, the *_TAG and
# CACHE_* paths, ...). This cell only pulls in the libraries used throughout;
# concept-specific helpers are imported inline at first use.
import csv
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from ase.io import read as ase_read
from loguru import logger

### Notebook-local utility

One small helper bridges the cached pathway to the analysis functions:

- **`ase_frames_to_batches`** maps `ase.io.read(path, index=':')` output to a list of single-graph `Batch` objects ready for the analysis helpers — a thin wrapper around `AtomicData.from_atoms` plus zero-allocated dynamics fields, so each frame is a self-contained, dynamics-ready record. The saved path then feeds the diagnostics indistinguishably from a live run.

Resolving a stage stem to its shipped files is handled by the `Config` object, keeping the path plumbing out of the notebook: `cfg.cached_extxyz(stem)` returns `data/cached/<RUN_NAME>/traj/<stem>.extxyz` (or `None`) and `cfg.cached_log_csv(stem)` the matching CSV under `csv/`.

(`AtomicData` and `Batch` are introduced in the next section, where we build the first one from a real CIF.)

In [ ]:
# ─── Notebook-local utility ────────────────────────────────────────────────
# AtomicData / Batch are the toolkit's two core data containers (introduced in
# the next section). AtomicData.from_atoms is the canonical ASE → toolkit
# converter — it reads positions / atomic_numbers / cell / pbc off any ASE
# Atoms object. (Stage-stem path resolution lives on cfg: cfg.cached_extxyz /
# cfg.cached_log_csv.)
from nvalchemi.data import AtomicData, Batch


def ase_frames_to_batches(frames, device="cpu"):
    """Convert an iterable of ase.Atoms into single-graph Batch objects ready
    for the analysis helpers. Velocities are zero — cached frames carry no
    momenta and none of the post-hoc analysers consume them."""
    batches = []
    for atoms in frames:
        data = AtomicData.from_atoms(atoms)
        n = data.num_nodes
        data.forces = torch.zeros(n, 3)
        data.energy = torch.zeros(1, 1)
        data.stress = torch.zeros(1, 3, 3)
        data.add_node_property("velocities", torch.zeros(n, 3))
        data.charge = torch.zeros(1, 1)
        batches.append(Batch.from_data_list([data], device=device))
    return batches

## The naphthalene crystal

Naphthalene ($\mathrm{C_{10}H_8}$) is the simplest fused-aromatic molecule and a textbook monoclinic molecular crystal — a representative stand-in for the planar aromatic cores at the heart of organic-semiconductor and OLED materials. It crystallises in space group $P2_1/a$ (#14) with $Z = 2$ molecules per unit cell in a herringbone motif. We start from the room-temperature reference structure of Brock & Dunitz (CCDC 1216816, refcode `NAPHTA10`; [1982](https://doi.org/10.1107/S0567740882008358)), which ASE reads straight from its CIF.

### `AtomicData` and `Batch` — the toolkit's data containers

Two core toolkit structures appear in every cell that touches positions or forces:

- **`AtomicData`** is the toolkit's analogue of ASE's `Atoms` — a single-system record holding `positions`, `atomic_numbers`, `cell`, `pbc`, and the per-step dynamics fields (`forces`, `energy`, `stress`, `velocities`). Conversion from ASE is one call, `AtomicData.from_atoms(atoms)`, which populates everything topology-related and leaves the dynamics fields for the caller to allocate.
- **`Batch`** collates one or more `AtomicData` records into the multi-graph object every integrator and hook consumes. A single system is wrapped in a length-1 `Batch` (our supercell below); later, the coexistence sweep packs all four temperatures into one `Batch` with a per-graph temperature tensor and advances them together in a single GPU launch.

We build the first one from real data next: read the CIF, tile a supercell, then construct `AtomicData` + `Batch`. `AtomicData.from_atoms` fills in the topology; we pre-allocate the dynamics fields (`forces` / `energy` / `stress` / `velocities`) as zeros for the integrator to write into in place, then move the whole `Batch` onto the device in a single step.

In [ ]:
# ─── Read the experimental crystal ─────────────────────────────────────────
# cfg.activate() already silenced ASE 3.28's cosmetic 'monoclinic' CIF warning
# (NAPHTA10 supplies explicit symmetry operators, so the setting is unambiguous).
unit_cell = ase_read("data/naphthalene.cif")
ATOMS_PER_MOL = len(unit_cell) // 2  # Z = 2 for P2_1/a

a, b, c = unit_cell.cell.lengths()
alpha, beta, gamma = unit_cell.cell.angles()
print(f"Unit cell: {len(unit_cell)} atoms ({ATOMS_PER_MOL} atoms/molecule)")
print(f"Cell: a={a:.3f}  b={b:.3f}  c={c:.3f} Å   β={beta:.1f}°")

In [ ]:
# Tile the unit cell into the SUPERCELL set in the configuration cell.
supercell = unit_cell * SUPERCELL
N_MOL = len(supercell) // ATOMS_PER_MOL
n_atoms_total = len(supercell)
print(f"Supercell {SUPERCELL}: {n_atoms_total} atoms ({N_MOL} molecules)")

### A density helper

Per-graph **density** (g/cm³) recurs in every diagnostic that follows — it is the cleanest single signal that the cell has equilibrated and, later, that a phase has melted. `nvalchemi` stores atomic masses in amu and cell vectors in Å, so the only subtlety is the unit conversion (1 amu/Å³ = 1.66054 g/cm³) and summing mass per graph (so the same helper works whether a `Batch` holds one system or several). We import the packaged `compute_density` rather than re-deriving it here.

In [ ]:
from helpers import compute_density

In [ ]:
# Build AtomicData from the ASE supercell, pre-allocate its dynamics fields, and
# pack it into a length-1 Batch on the device (see above).
data = AtomicData.from_atoms(supercell)
data.forces = torch.zeros(n_atoms_total, 3)
data.energy = torch.zeros(1, 1)
data.stress = torch.zeros(1, 3, 3)
data.add_node_property("velocities", torch.zeros(n_atoms_total, 3))
data.charge = torch.zeros(1, 1)
batch = Batch.from_data_list([data], device=DEVICE)

cell_lengths = batch.cell.squeeze().norm(dim=-1)
print(f"Cell lengths: {[f'{x:.1f}' for x in cell_lengths.tolist()]} Å")
print(f"Density:      {compute_density(batch)[0]:.3f} g/cm³  (experimental ≈ 1.18 at 298 K)")

# Each cell vector must exceed twice the model's neighbour cutoff (Orb-v3 uses 6 Å).
assert (cell_lengths > 12.0).all(), "supercell too small for the 6 Å model cutoff"

In [ ]:
# ─── Visualise the starting crystal (OVITO) ────────────────────────────────
# display_widgets_row renders a labelled row of interactive 3-D OVITO widgets
# (drag to rotate, scroll to zoom) — the same viewer used throughout the
# notebook for structures and trajectories. We pass the ASE supercell directly
# and show the monoclinic cell wireframe.
from helpers import display_widgets_row

display_widgets_row(
    [(f"naphthalene {SUPERCELL[0]}×{SUPERCELL[1]}×{SUPERCELL[2]} — {N_MOL} molecules", supercell)],
    width="460px",
    height="400px",
    show_cell=True,
)

## A foundation model: building a custom Orb-v3 wrapper

The Toolkit ships ready-made wrappers for some machine-learned potentials (e.g. `MACEWrapper`, `AIMNet2Wrapper`). Our model — **Orb-v3**, a conservative MLIP from Orbital Materials, in its **OMol** flavour trained for molecular chemistry — isn't one of them. That is the common case in research: the model you want is often newer than the wrappers any framework ships.

The Toolkit is built for exactly this. Its `BaseModelMixin` contract defines a small surface that *any* MLIP can implement to plug into the integrators, hooks, and pipelines. We build that wrapper from scratch — it is short, and seeing it makes the rest of the notebook (and wrapping your own models) transparent. The job is narrow: consume a Toolkit `Batch`, and return `{energy, forces, stress}` for every system in it. (Orb-v3: [arXiv:2504.06231](https://arxiv.org/abs/2504.06231); code: [orb-models](https://github.com/orbital-materials/orb-models).)

### The model contract

A wrapper advertises what it computes and what it needs through a `ModelConfig`:

- **`active_outputs`** — the quantities each forward returns. We need `{energy, forces, stress}`; the `stress` is what later lets the anisotropic-NPT barostat see a real per-axis pressure.
- **`supports_pbc` / `needs_pbc`** — Orb-v3 is a periodic model.
- **`neighbor_config`** — a `NeighborConfig(cutoff=6 Å, format=COO, skin=0)`. Declaring this is the key trick: the Toolkit then auto-installs a `NeighborListHook` that **populates the neighbour list on the batch before every forward**, so the wrapper just *reads* `batch.neighbor_list` instead of rebuilding it. The same `make_neighbor_hooks()` plugs into the safety chain we use for MD.

### Framework → model: building Orb's `AtomGraphs` (`adapt_input`)

Orb consumes its own `AtomGraphs` object, so the conversion reads the hook-populated neighbour list off the `Batch` and assembles it:

- **edge vectors** $\vec r_j - \vec r_i + (\mathbf{s}\cdot\mathbf{cell})$ from the sender/receiver indices and the integer periodic shifts $\mathbf{s}$;
- a **one-hot** encoding of atomic numbers (via a cached identity table, cheaper per step than allocating a fresh array);
- the **OMol conditioning** fields `total_charge = 0` and `spin_multiplicity = 1` — the neutral closed-shell case, correct for naphthalene and the other neutral molecular crystals here.

### Model → framework: energy, forces, stress (`adapt_output`)

Orb returns analytical gradients, which we map back to the Toolkit's shapes:

- **forces** and **stress** are renamed from Orb's `grad_forces` / `grad_stress`;
- **stress** is converted from Orb's **Voigt-6** vector `[xx, yy, zz, yz, xz, xy]` to the symmetric `3×3` tensor every integrator expects;
- the **energy is detached** — Orb already differentiated it internally to produce its conservative forces and stress, so we return the value without its (freed) autograd graph;
- **sign convention**: Orb and the Toolkit's NPT integrator both use **tension-positive** Cauchy stress, so no flip is applied.

The wrapper also declares `forces` and `stress` as *direct derivative keys* — a small hint that these are analytical model outputs, not quantities the framework should obtain by differentiating through the wrapper.

In [ ]:
# ─── A from-scratch Orb-v3 wrapper ──────────────────────────────────────────
from torch import nn
from orb_models.forcefield import pretrained
from nvalchemi.models.base import (
    BaseModelMixin,
    ModelConfig,
    NeighborConfig,
    NeighborListFormat,
)

_NUM_ATOMIC_CLASSES = 118  # one-hot encoding spans the periodic table (Z = 1..118)


def voigt6_to_3x3(v):
    """Voigt-6 stress [xx, yy, zz, yz, xz, xy] -> symmetric 3x3."""
    xx, yy, zz, yz, xz, xy = v.unbind(-1)
    return torch.stack(
        [
            torch.stack([xx, xy, xz], dim=-1),
            torch.stack([xy, yy, yz], dim=-1),
            torch.stack([xz, yz, zz], dim=-1),
        ],
        dim=-2,
    )


class OrbV3Wrapper(nn.Module, BaseModelMixin):
    """A from-scratch Toolkit wrapper around an Orb-v3 conservative force field."""

    def __init__(self, orbff, atoms_adapter, device="cuda"):
        super().__init__()
        self.orbff = orbff
        self.atoms_adapter = atoms_adapter
        self._device = torch.device(device)
        self._radius = float(atoms_adapter.radius)
        self._max_neighbors = int(atoms_adapter.max_num_neighbors)
        self._dtype = next(orbff.parameters()).dtype
        self._pbc_row = torch.tensor([True, True, True], device=self._device)
        # Pre-built one-hot lookup over the periodic table (cheaper per step
        # than re-allocating a fresh (N, 118) tensor every forward).
        self.register_buffer(
            "_node_emb",
            torch.eye(_NUM_ATOMIC_CLASSES, dtype=self._dtype, device=self._device),
            persistent=False,
        )
        self.model_config = ModelConfig(
            outputs=frozenset({"energy", "forces", "stress"}),
            autograd_outputs=frozenset(),
            autograd_inputs=frozenset(),
            required_inputs=frozenset(),
            optional_inputs=frozenset(),
            supports_pbc=True,
            needs_pbc=True,
            neighbor_config=NeighborConfig(
                cutoff=self._radius,
                format=NeighborListFormat.COO,
                half_list=False,
                skin=0.0,
            ),
            active_outputs={"energy", "forces", "stress"},
        )

    def direct_derivative_keys(self):
        # Orb supplies analytical forces/stress; don't differentiate through us.
        return {"forces", "stress"}

    @property
    def embedding_shapes(self):
        mm = self.orbff.model
        dim = getattr(mm, "node_embed_size", None) or getattr(mm, "latent_dim", 256)
        return {"node_embeddings": (int(dim),), "graph_embeddings": (int(dim),)}

    def compute_embeddings(self, data, **kwargs):
        # Node/graph embeddings are part of the contract but unused by the MD here.
        raise NotImplementedError("embeddings are not used in this melting-point workflow")

    def _build_atom_graphs(self, data):
        from orb_models.common.atoms.batch.graph_batch import AtomGraphs

        dev, dt = self._device, self._dtype
        positions = data.positions.to(dtype=dt, device=dev).contiguous()
        atomic_numbers = data.atomic_numbers.to(dtype=torch.long, device=dev)
        cells = data.cell.to(dtype=dt, device=dev)
        batch_idx = data.batch_idx.to(device=dev)
        n_graphs = data.num_graphs

        # Neighbour list populated by the Toolkit's NeighborListHook.
        nl = data.neighbor_list.to(device=dev)
        unit_shifts = data.neighbor_list_shifts.to(device=dev).to(dtype=dt)
        senders, receivers = nl[:, 0].long(), nl[:, 1].long()
        edge_graph = batch_idx[senders]
        shifts_cart = torch.einsum("ei,eij->ej", unit_shifts, cells[edge_graph])
        vectors = positions[receivers] - positions[senders] + shifts_cart

        return AtomGraphs(
            senders=senders,
            receivers=receivers,
            n_node=torch.bincount(batch_idx, minlength=n_graphs),
            n_edge=torch.bincount(edge_graph, minlength=n_graphs).to(device=dev),
            node_features={
                "positions": positions,
                "atomic_numbers": atomic_numbers,
                "atomic_numbers_embedding": self._node_emb.index_select(0, atomic_numbers),
            },
            edge_features={"vectors": vectors, "unit_shifts": unit_shifts},
            system_features={
                "cell": cells,
                "pbc": self._pbc_row.unsqueeze(0).expand(n_graphs, -1).contiguous(),
                "total_charge": torch.zeros(n_graphs, dtype=dt, device=dev),
                "spin_multiplicity": torch.ones(n_graphs, dtype=dt, device=dev),
            },
            node_targets={},
            edge_targets={},
            system_targets={},
            system_id=None,
            fix_atoms=None,
            tags=None,
            radius=self._radius,
            max_num_neighbors=torch.full(
                (n_graphs,), self._max_neighbors, dtype=torch.long, device=dev
            ),
        )

    def adapt_input(self, data, **kwargs):
        if isinstance(data, AtomicData):
            data = Batch.from_data_list([data])
        return {"atom_graphs": self._build_atom_graphs(data), "num_graphs": data.num_graphs}

    def adapt_output(self, model_output, data):
        n_graphs = data.num_graphs if isinstance(data, Batch) else 1
        energy = model_output["energy"].detach().reshape(n_graphs, 1)
        forces = model_output[self.orbff.grad_forces_name]
        stress = voigt6_to_3x3(torch.atleast_2d(model_output[self.orbff.grad_stress_name]))
        return super().adapt_output({"energy": energy, "forces": forces, "stress": stress}, data)

    def forward(self, data, **kwargs):
        graphs = self.adapt_input(data, **kwargs)["atom_graphs"]
        return self.adapt_output(self.orbff.predict(graphs), data)

    @classmethod
    def from_checkpoint(cls, alias="orb_v3_conservative_omol", device="cuda",
                        precision="float32-high", compile_model=False):
        orbff, adapter = getattr(pretrained, alias)(
            device=device, precision=precision, compile=compile_model
        )
        return cls(orbff, adapter, device=device)

### Loading the checkpoint and a single-point check

`from_checkpoint` downloads the `orb_v3_conservative_omol` weights (first run only) and wraps them. We use the **vanilla** model with no separate dispersion correction: the OMol checkpoint is trained on a dispersion-inclusive functional, so the van der Waals interactions that hold a molecular crystal together are already in the potential. (Composing an explicit Grimme-D3 term on top is demonstrated in the Part 3 tutorial.)

We then run one **single-point** on the crystal `Batch` to confirm the wrapper is wired correctly — it should return `energy`, `forces`, and a **symmetric `stress`** tensor. That last check matters here: the coexistence stage drives an anisotropic barostat off this stress, so a model that silently produced none would quietly break the melting-point result.

In [ ]:
# ─── Load Orb-v3 (OMol) and validate with a single-point ────────────────────
import types

from nvalchemi.dynamics.base import DynamicsStage

# Vanilla Orb-v3 (OMol): trained on a dispersion-inclusive functional, so no
# separate D3 correction is needed here.
model = OrbV3Wrapper.from_checkpoint(
    "orb_v3_conservative_omol",
    device=DEVICE,
    precision="float32-high",
    compile_model=COMPILE_MODEL,
)

# Populate the neighbour list (as the safety chain will every MD step), run one
# forward, and confirm the three outputs are sane.
for hook in model.make_neighbor_hooks():
    hook(types.SimpleNamespace(batch=batch), DynamicsStage.BEFORE_COMPUTE)
out = model(batch)
stress = out["stress"].reshape(3, 3)
assert sorted(out) == ["energy", "forces", "stress"]
assert torch.isfinite(out["energy"]).all() and torch.isfinite(out["forces"]).all()
assert torch.allclose(stress, stress.T, atol=1e-4)
print(f"energy        {out['energy'].reshape(-1)[0].item():.1f} eV  ({n_atoms_total} atoms)")
print(f"forces        {tuple(out['forces'].shape)}  |max| = {out['forces'].abs().max().item():.2f} eV/Å")
print(f"stress (3×3)  symmetric, trace = {stress.trace().item():.4f} eV/Å³")

### How much does batching buy?

The wrapper returns the same energy / forces / stress whether it sees one system or many — and the Toolkit's real value is that it can evaluate *many independent systems in a single `Batch`*. A GPU launch carries a fixed cost (kernel dispatch, neighbour-list setup, host/Python overhead) that batching amortizes across every system in the batch.

We measure it directly: pack 1, 2, 4, … independent naphthalene unit cells into one `Batch`, run a short relaxation, and compare the wall time to running them one at a time. It is a quick live benchmark — a few seconds of short FIRE2 relaxations — and each batch shape is warmed once so the timing reflects steady-state throughput rather than first-call kernel setup.

In [ ]:
# ─── Batched throughput: pack N unit cells into one Batch and time it ───────
from time import perf_counter

from nvalchemi.dynamics.hooks import NaNDetectorHook
from nvalchemi.dynamics.optimizers import FIRE2

from helpers import NotebookProgress, plot_batch_speedup

BENCH_SIZES = [1, 2, 4, 8, 16]  # independent naphthalene unit cells per Batch
BENCH_STEPS = 10                     # short FIRE2 relaxation per timing run


def time_orb_batch(n_cells):
    """Wall time for a short FIRE2 relaxation of n_cells independent unit cells."""
    datas = []
    for _ in range(n_cells):
        d = AtomicData.from_atoms(unit_cell, device="cpu", dtype=DTYPE)
        d.forces = torch.zeros_like(d.positions)
        d.energy = torch.zeros(1, 1, dtype=DTYPE)
        d.stress = torch.zeros(1, 3, 3, dtype=DTYPE)
        datas.append(d)
    batch_n = Batch.from_data_list(datas, device=DEVICE)
    opt = FIRE2(model, dt=0.01, n_steps=BENCH_STEPS)
    for hook in model.make_neighbor_hooks():
        opt.register_hook(hook)
    opt.register_hook(NaNDetectorHook(extra_keys=["stress"]))
    if str(DEVICE).startswith("cuda"):
        torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    opt.run(batch_n)
    if str(DEVICE).startswith("cuda"):
        torch.cuda.synchronize(DEVICE)
    return perf_counter() - start


progress = NotebookProgress(
    title="Orb-v3 batched throughput",
    total=len(BENCH_SIZES),
    unit="batch sizes",
    message="timing short FIRE2 relaxations",
    average_label="s/size",
)
times = {}
for i, size in enumerate(BENCH_SIZES, start=1):
    progress.update(done=i - 1, message=f"batch of {size} unit cells")
    time_orb_batch(size)                 # warm this batch shape (one-time setup)
    times[size] = time_orb_batch(size)   # timed
    progress.update(done=i, message=f"batch of {size}: {times[size]:.3f} s")

single = times[BENCH_SIZES[0]]
speedups = [(single * size) / times[size] for size in BENCH_SIZES]
plot_batch_speedup(
    BENCH_SIZES,
    speedups,
    system_label="naphthalene unit cells",
    suptitle="Orb-v3 batched throughput on the ALCHEMI Toolkit",
)
print(f"Largest batch ({BENCH_SIZES[-1]} cells): {speedups[-1]:.1f}× vs one-at-a-time")

## Warmup: FIRE2 → NVT → anisotropic NPT

Before we can melt anything we need a properly equilibrated crystal. Three sub-stages take the experimental supercell from its as-read geometry to a thermalised, density-relaxed configuration at the warmup temperature ($T_\mathrm{warmup} = $ `T_WARMUP` K):

- **Stage A — FIRE2 minimisation.** A fast inertial optimiser drives the per-atom forces below `FMAX` eV/Å, cleaning up the small clashes introduced by tiling the unit cell.
- **Stage B — NVT thermalisation.** Maxwell–Boltzmann velocities are seeded at $T_\mathrm{warmup}$ and a Langevin thermostat (friction `FRICTION` fs⁻¹) settles the kinetic-energy distribution into thermal equilibrium.
- **Stage C — anisotropic NPT.** A Martyna–Tobias–Klein barostat (τ_P = `BAROSTAT_TIME` fs) holds 1 atm on each diagonal cell axis while a Nosé–Hoover chain thermostat (τ_T = `THERMOSTAT_TIME` fs) holds $T_\mathrm{warmup}$, letting the monoclinic lattice constants relax to the model's preferred density.

**Cost vs. caching.** Stages A and B are cheap; with `RESULT_SOURCE="compute"` we run FIRE2 (capped at `LIVE_FIRE_MAX_STEPS` steps) and a short (`LIVE_NVT_PS` ps) NVT live so you can watch the pipeline work. The ~50 ps NPT equilibration is the expensive part — it is **always shown from the shipped cache** (a pre-rendered figure plus an animation rendered live with OVITO from the cached trajectory), never re-run in the notebook. With the default `RESULT_SOURCE="saved"` every stage is shown from cache.

### Why anisotropic NPT?

A scalar pressure target with `pressure_coupling="isotropic"` drives all three diagonal cell components toward a single hydrostatic value — fine for a cubic crystal, but it imposes spurious lateral strain on a low-symmetry cell. Naphthalene is monoclinic ($\beta \approx 123.6°$), so its $a$, $b$, $c$ axes equilibrate to *different* lengths under 1 atm. Anisotropic coupling lets each axis follow its own component of the stress tensor:

$$P_\mathrm{target} = \begin{bmatrix} P_0 & P_0 & P_0 \end{bmatrix}, \qquad P_0 = 1~\mathrm{atm}.$$

In `nvalchemi` this is passed as a **rank-2 `[1, 3]` tensor** (`torch.tensor([[P, P, P]])`) that broadcasts to every graph in the batch. A *scalar* `P` with `pressure_coupling="anisotropic"` silently falls through to isotropic coupling, so the tensor shape matters. Pressure is in **eV/Å³** (matching the model's stress output) and friction in **fs⁻¹** — `P_1ATM` from `helpers.constants` does the unit conversion.

Anisotropic coupling is also **mandatory** for the solid–liquid coexistence stage later: there the interface-normal axis must move independently of the in-plane axes that are pinned by the solid lattice.

### Hooks — the toolkit's lifecycle callback system

Most of what follows is built from **hooks**: callbacks that fire at specific points in an integrator's per-step lifecycle (`BEFORE_STEP`, `BEFORE_COMPUTE`, `AFTER_STEP`, `AFTER_POST_UPDATE`, `ON_CONVERGE`, …). The safety chain, per-step CSV/progress logging, snapshot writing, and even FIRE2's fmax termination criterion are all hooks. You attach them after constructing the integrator:

```python
stage = NVTLangevin(model=model, ...)
for hook in [*make_safety_hooks(model), logging_hook]:
    stage.register_hook(hook)
```

**Order matters** when one hook feeds the next — the neighbour-list rebuild must precede the max-force clamp so the clamp sees fresh edges. The next two cells set up the two recurring hook patterns we reuse across the warmup, liquid-half, and SLC pipelines.

### The safety-hook chain

`make_safety_hooks(model)` (from `helpers`) assembles the defensive chain every integrator below runs with, in order:

1. **model-supplied neighbour-list rebuild** — different models need different neighbour logic, so we delegate to `model.make_neighbor_hooks()`;
2. **periodic-boundary wrapping** (`WrapPeriodicHook`, fired after each position update);
3. **max-force clamp** (`MaxForceClampHook`) — guards against rare forward-pass spikes;
4. **NaN detector** (`NaNDetectorHook`) — fails fast rather than propagating garbage.

The `track_stress` flag toggles whether the NaN detector also watches `stress`: turn it **off** for FIRE2 (no autograd through the stress tape) and **on** for NVT/NPT (where the barostat consumes stress).

In [ ]:
from nvalchemi.dynamics.base import DynamicsStage  # noqa: F401  (referenced in markdown)

from helpers import make_safety_hooks

# The chain assembled for an NVT/NPT stage (stress-tracking on):
for hook in make_safety_hooks(model, track_stress=True):
    print(f"  {type(hook).__name__:<22} stage={getattr(hook, 'stage', '—')}")

### `LoggingHook` and its custom-scalar protocol

`LoggingHook` records per-step quantities into one of two backends: **`backend="csv"`** writes one row per (step, graph) to disk, and **`backend="custom"`** invokes a user-supplied `writer_fn(step, rows)`. In Jupyter we use the custom backend to drive a `NotebookProgress` bar via `make_progress_writer` — each step's scalars tick a live progress strip in the cell output. The default columns are `step`, `graph_idx`, `status`, `energy`, `fmax`, `temperature`.

On top of the defaults we attach three **custom scalars** — pressure, volume, density — bundled in `DYNAMICS_SCALARS` (from `helpers.dynamics`). Each is a function `(ctx) -> Tensor[num_graphs]` returning one value per graph every time the hook fires. We also import `P_1ATM` (1 atm in eV/Å³) for the NPT target.

> Note on the pressure scalar: `nvalchemi` uses a **tension-positive Cauchy** convention for `batch.stress` ($\sigma = -W/V$), so the scalar pressure is $P = \tfrac{2}{3}\,\mathrm{KE}/V - \tfrac{1}{3}\,\mathrm{Tr}(\sigma)$. A compressed crystal therefore reports a *negative* diagonal stress — there is no sign flip anywhere in the pipeline.

In [ ]:
from helpers import DYNAMICS_SCALARS, P_1ATM, NotebookProgress, make_progress_writer

print(f"Custom log scalars: {', '.join(DYNAMICS_SCALARS)}")
print(f"P_1ATM = {P_1ATM:.3e} eV/Å³")

### What runs live, and the plumbing for it

In `compute` mode the live stages write a Zarr trajectory and a CSV log under `LOG_DIR` so the diagnostics in the next section can read them back. We import three small file-IO helpers for that:

| Helper | Purpose |
|--------|---------|
| `fresh_zarr_sink(path, capacity)` | a pre-sized `ZarrData` snapshot sink (wipes any stale store first) |
| `save_checkpoint(batch, name, LOG_DIR)` / `load_checkpoint(...)` | persist / restore an end-of-stage `Batch` |
| `SnapshotHook` / `LoggingHook` | the per-step trajectory + log writers introduced above |

The full driver (`warmup.py`) adds resume/extend bookkeeping (multi-part logs, integrator-state reload); the notebook keeps things linear — it runs the short live stages **fresh** and reads everything heavy from the cache.

In [ ]:
from nvalchemi.dynamics import initialize_velocities
from nvalchemi.dynamics.base import ConvergenceHook
from nvalchemi.dynamics.hooks import LoggingHook, SnapshotHook
from nvalchemi.dynamics.integrators.npt import NPT
from nvalchemi.dynamics.integrators.nvt_langevin import NVTLangevin
from nvalchemi.dynamics.optimizers.fire2 import FIRE2

from helpers import fresh_zarr_sink, load_checkpoint, save_checkpoint

# Clean cached stems (the shipped cache + staging tool agree on these names).
FIRE_STEM = "warmup_fire"
NVT_STEM = "warmup_nvt"
NPT_STEM = "warmup_npt"

# Live step budgets for compute mode. FIRE2 terminates early via its fmax
# convergence hook; LIVE_FIRE_MAX_STEPS is just the safety cap. The cached
# canonical run used FIRE_MAX_STEPS as the FIRE cap and thermalised for
# WARMUP_NVT_PS ps; live mode runs a short LIVE_NVT_PS-ps NVT so the pipeline
# is observable without a long wait.
N_NVT_LIVE = int(LIVE_NVT_PS * 1000 / DT)  # LIVE_NVT_PS ps in steps at DT fs

print(f"Live FIRE2 cap : {LIVE_FIRE_MAX_STEPS} steps (fmax < {FMAX} eV/Å terminates early)   [cached run used {FIRE_MAX_STEPS}]")
print(f"Live NVT       : {LIVE_NVT_PS:g} ps  ({N_NVT_LIVE} steps)   [cached run used {WARMUP_NVT_PS:g} ps]")
print(f"Cached NPT     : {WARMUP_NPT_PS:g} ps  (shown from cache, never run live)")

### Stage A — FIRE2 minimisation

Drive the per-atom forces below `FMAX` eV/Å. FIRE2 is *inertial pseudo-dynamics*: an internal `dt` ≈ 0.01 (unrelated to the physical MD timestep) couples velocity to force with an adaptive step size, so never feed it the physical `DT`. We disable stress tracking in `make_safety_hooks` here — FIRE2 does not propagate autograd through the stress tape — and wire FIRE2's fmax termination via `ConvergenceHook.from_fmax`, the same hook-composition mechanic as everything else. In `compute` mode we cap the live run at `LIVE_FIRE_MAX_STEPS` steps (the cached canonical run used `FIRE_MAX_STEPS`); convergence almost always terminates well before the cap.

In [ ]:
if USE_SAVED:
    print("RESULT_SOURCE='saved': skipping live FIRE2 — see the cached results below.")
else:
    logger.info("[FIRE] start (≤ {} steps; fmax < {} eV/Å terminates early)", LIVE_FIRE_MAX_STEPS, FMAX)
    fire_zarr = fresh_zarr_sink(
        LOG_DIR / f"{FIRE_STEM}.zarr", capacity=LIVE_FIRE_MAX_STEPS // SNAPSHOT_EVERY + 10
    )
    fire_csv = LoggingHook(
        backend="csv",
        custom_scalars=DYNAMICS_SCALARS,
        log_path=str(LOG_DIR / f"{FIRE_STEM}.csv"),
        frequency=LOG_EVERY,
    )
    fire_progress = NotebookProgress(
        title="FIRE2 minimisation",
        total=LIVE_FIRE_MAX_STEPS,
        unit="steps",
        message="relaxing forces",
        average_label="s/step",
    )
    fire_out = LoggingHook(
        backend="custom",
        writer_fn=make_progress_writer(fire_progress),
        custom_scalars=DYNAMICS_SCALARS,
        frequency=LOG_EVERY,
    )
    fire_stage = FIRE2(
        model=model,
        dt=0.01,
        n_steps=LIVE_FIRE_MAX_STEPS,
        convergence_hook=ConvergenceHook.from_fmax(threshold=FMAX),
    )
    for hook in [
        *make_safety_hooks(model, track_stress=False),
        SnapshotHook(sink=fire_zarr, frequency=SNAPSHOT_EVERY),
        fire_csv,
        fire_out,
    ]:
        fire_stage.register_hook(hook)
    with fire_csv, fire_out:
        batch = fire_stage.run(batch)
    fire_progress.update(done=LIVE_FIRE_MAX_STEPS, message="converged")
    save_checkpoint(batch, FIRE_STEM, LOG_DIR)
    print(f"FIRE2 done: fmax = {batch.forces.norm(dim=-1).max().item():.4f} eV/Å")

### Stage B — NVT thermalisation

Seed Maxwell–Boltzmann velocities at `T_WARMUP` (`initialize_velocities` handles per-graph mass weighting, COM removal, and KE rescaling), then run Langevin NVT. In `compute` mode we run a short **`LIVE_NVT_PS` ps** burst (`N_NVT_LIVE` steps) — long enough to see the temperature settle and the density hold; the shipped cache was thermalised for `WARMUP_NVT_PS` ps. Langevin dynamics is *memoryless*, so there is no integrator state to persist between calls.

In [ ]:
if USE_SAVED:
    print("RESULT_SOURCE='saved': skipping live NVT — see the cached results below.")
else:
    logger.info("[NVT] start ({} steps @ {} K)", N_NVT_LIVE, T_WARMUP)
    batch.velocities = torch.zeros_like(batch.positions)
    initialize_velocities(
        batch.velocities,
        batch.atomic_masses,
        temperature=torch.tensor([T_WARMUP], device=DEVICE),
        batch_idx=batch.batch_idx,
        random_seed=42,
        remove_com=True,
        rescale=True,
    )
    nvt_zarr = fresh_zarr_sink(
        LOG_DIR / f"{NVT_STEM}.zarr", capacity=N_NVT_LIVE // SNAPSHOT_EVERY + 10
    )
    nvt_csv = LoggingHook(
        backend="csv",
        custom_scalars=DYNAMICS_SCALARS,
        log_path=str(LOG_DIR / f"{NVT_STEM}.csv"),
        frequency=LOG_EVERY,
    )
    nvt_progress = NotebookProgress(
        title="NVT thermalisation",
        total=N_NVT_LIVE,
        unit="steps",
        message=f"thermalising @ {int(T_WARMUP)} K",
        average_label="s/step",
    )
    nvt_out = LoggingHook(
        backend="custom",
        writer_fn=make_progress_writer(nvt_progress),
        custom_scalars=DYNAMICS_SCALARS,
        frequency=LOG_EVERY,
    )
    nvt_stage = NVTLangevin(
        model=model,
        dt=DT,
        temperature=T_WARMUP,
        friction=FRICTION,
        n_steps=N_NVT_LIVE,
    )
    for hook in [
        *make_safety_hooks(model),
        SnapshotHook(sink=nvt_zarr, frequency=SNAPSHOT_EVERY),
        nvt_csv,
        nvt_out,
    ]:
        nvt_stage.register_hook(hook)
    with nvt_csv, nvt_out:
        batch = nvt_stage.run(batch)
    nvt_progress.update(done=N_NVT_LIVE, message="done")
    save_checkpoint(batch, NVT_STEM, LOG_DIR)
    print(
        f"NVT done: density = {compute_density(batch)[0]:.3f} g/cm³  "
        f"(cell held fixed in NVT)"
    )

### Stage C — anisotropic NPT (shown from cache)

The final stage couples a Martyna–Tobias–Klein barostat (τ_P = `BAROSTAT_TIME` fs) at 1 atm to a Nosé–Hoover chain thermostat (τ_T = `THERMOSTAT_TIME` fs) at `T_WARMUP`, with the `[1, 3]` pressure tensor from above. The cell relaxes anisotropically over ~`WARMUP_NPT_PS` ps until the lattice constants and density converge.

This stage is **never run live in the notebook** — ~50 ps of NPT on 3600 atoms is the most expensive single step in the warmup, and re-running it adds nothing the cache cannot show. Below we render the equilibration figure (energy, temperature, density, and cell lengths vs. time) live from the cached log, alongside an animation rendered live with OVITO from the cached trajectory of the relaxing crystal. In `compute` mode you have already produced a fresh FIRE2 + `LIVE_NVT_PS`-ps NVT checkpoint above; the production NPT result still comes from the cache.

> For full reproducibility, the production driver (`warmup.py`) persists the NPT integrator's internal `_state` (Nosé–Hoover chains + barostat momenta) so a long run can be resumed transient-free — a detail worth knowing if you extend this run yourself.

In [ ]:
from IPython.display import display

from helpers import display_trajectory_animation, plot_warmup_stage

# Equilibration diagnostics, rendered live from the shipped NPT log (no pre-baked PNG).
npt_csv = cfg.cached_log_csv("warmup_npt")
npt_traj = cfg.cached_extxyz("warmup_npt")

if npt_csv is not None:
    fig = plot_warmup_stage(
        npt_csv, stage="npt", t_warmup=T_WARMUP, dt=DT, lattice_extxyz=npt_traj
    )
    display(fig)
    plt.close(fig)
else:
    print("[cache pending] warmup_npt log not found — run tools/stage_cached_run.py (task #14).")

# Render the cached NPT trajectory live with OVITO (no shipped MP4).
if npt_traj is not None:
    frames = ase_read(str(npt_traj), index=":")
    display_trajectory_animation(
        frames,
        label=f"naphthalene crystal · NPT @ {int(T_WARMUP)} K",
        target_frames=ANIM_TARGET_FRAMES,
    )
else:
    print("[cache pending] warmup_npt trajectory not found — run tools/stage_cached_run.py (task #14).")

The warmup is complete: the crystal is minimised, thermalised, and density-relaxed at `T_WARMUP` K and 1 atm. The model settles naphthalene to ≈ 1.18 g/cm³ — within a percent of the experimental room-temperature density ([Brock & Dunitz, 1982](https://doi.org/10.1107/S0567740882008358)) — a first sanity check that Orb-v3 reproduces the right condensed-phase packing.

In the next section we load the warmup trajectory back and read it quantitatively: energy/temperature/density convergence, the centre-of-mass diffusion coefficient $D$, and the rotational order parameter $S_0$. Together $D \approx 0$ and $S_0 \to 1$ confirm the endpoint is a well-ordered solid — the reference state for the melting-point screen.

## Warmup diagnostics

Before we melt anything, we need quantitative confirmation that the warmup actually *equilibrated a crystal* — not a defective or partially molten solid. We run three checks on the warmup-NPT endpoint:

1. **Density plateau** — the cell volume has settled and $\rho$ sits at the experimental value.
2. **Translational order** — the molecular centre-of-mass mean-squared displacement (COM-MSD) plateaus, so the diffusion coefficient $D \approx 0$.
3. **Rotational order** — the rotational order parameter $S_0 \to 1$, so molecules keep their lattice orientation.

Together, $D$ and $S_0$ form the **phase classifier** of Schmidt, Van der Spoel & Walz ([Schmidt et al., 2023](https://doi.org/10.1021/acsphyschemau.2c00045)) — the same procedure we will reuse to read out each coexistence temperature later. Applied to the warmup endpoint we want the unambiguous **crystal** signature: $D \approx 0$ **and** $S_0 \to 1$.

### The analysis functions

The heavy maths is in `helpers` so the notebook stays readable; here is what each does and why it is non-trivial under periodic boundaries and a fluctuating NPT cell:

- **`compute_com_msd(positions, cells, masses, atoms_per_mol)`** — the physically meaningful *translational* MSD. It (a) collapses each molecule to a mass-weighted centre of mass, PBC-unwrapped relative to its first atom (removing C–H vibrations and molecular rotation), (b) accumulates displacement in **fractional** coordinates using *each frame's own cell* (so an atom pinned to a fixed lattice site contributes zero MSD even while the NPT cell breathes), and (c) subtracts the mass-weighted system drift each step (`subtract_system_com=True`). Skipping (b)/(c) inflates a stable crystal's MSD by one–two orders of magnitude. Returns `[n_frames-1, n_mol]`.

- **`fit_diffusion_coefficient(msd_per_mol, time_ps, fit_frac=0.5)`** — fits the trailing 50% of the cross-molecule mean MSD to the 3-D Einstein relation $\mathrm{MSD} = 6 D t$, skipping the ballistic head. Returns $D$ in both Å²/ps and cm²/s.

- **`compute_S0_from_frames(frames, atoms_per_mol, ref_idx=0, tail_frac=0.2)`** — chains principal-inertia-axis extraction → second-rank ($P_2$) rotational autocorrelation vs the reference frame → tail average. Returns `(s0_per_mol [n_mol], s0_per_mol_per_axis [n_mol, 3], acf [n_frames, n_mol, 3])` — **per molecule**, so we average for the scalar read.

In [ ]:
# ─── Diagnostics imports + shared dark-figure style ────────────────────────
from helpers import compute_S0_from_frames, compute_com_msd, fit_diffusion_coefficient

# Cached warmup trajectory is thinned to one frame per CACHED_WARMUP_STEP_SPACING
# MD steps (see staging tool). At DT fs/step this sets the physical frame spacing.
CACHED_WARMUP_STEP_SPACING = 1000  # MD steps between cached warmup snapshots

_NV = {
    "green": "#76B900", "blue": "#00A3E0", "dark": "#000000",
    "light": "#F3F5F7", "muted": "#A8B0B8", "grid": "#2F3A44", "spine": "#4B5563",
}


def _style_dark_ax(ax, *, xlabel="", ylabel="", title=""):
    """Apply the NVIDIA dark palette to a matplotlib Axes."""
    ax.set_facecolor(_NV["dark"])
    ax.set_xlabel(xlabel, color=_NV["light"])
    ax.set_ylabel(ylabel, color=_NV["light"])
    if title:
        ax.set_title(title, color=_NV["light"], pad=10)
    ax.tick_params(colors=_NV["light"])
    for s in ax.spines.values():
        s.set_color(_NV["spine"])
    ax.grid(True, color=_NV["grid"], linewidth=0.8, alpha=0.7)

### NVT thermalisation, from the cached log

The NVT stage thermalised the minimised crystal at fixed cell: the temperature was driven up to the target while the energy and pressure settled around stable values. We render that stage live from the shipped NVT log below. (The NPT equilibration figure — temperature, density, and cell-vector relaxation — was already shown in the warmup section above.)

In [ ]:
# ─── NVT thermalisation figure, rendered live from the cached log ──────────
from helpers import plot_warmup_stage

nvt_csv = cfg.cached_log_csv("warmup_nvt")
if nvt_csv is not None:
    fig = plot_warmup_stage(nvt_csv, stage="nvt", t_warmup=T_WARMUP, dt=DT)
    display(fig)
    plt.close(fig)
else:
    print("[cache pending] warmup_nvt log not found — run tools/stage_cached_run.py.")

In [ ]:
# ─── Load the cached (thinned) warmup-NPT trajectory ───────────────────────
warmup_npt_path = cfg.cached_extxyz("warmup_npt")
if warmup_npt_path is None:
    raise FileNotFoundError(
        "Cached warmup-NPT trajectory data/cached/"
        f"{RUN_NAME}/traj/warmup_npt.extxyz is missing — stage the cache "
        "(tools/stage_cached_run.py) or run the full warmup yourself."
    )

warmup_frames = ase_frames_to_batches(ase_read(str(warmup_npt_path), index=":"), device="cpu")
snap_positions = [b.positions for b in warmup_frames]
snap_cells = [b.cell.squeeze() for b in warmup_frames]

dt_per_frame_ps = CACHED_WARMUP_STEP_SPACING * DT / 1000.0
time_ps = np.arange(len(warmup_frames)) * dt_per_frame_ps

print(
    f"Loaded {len(warmup_frames)} warmup-NPT frames "
    f"({snap_positions[0].shape[0]} atoms each), "
    f"Δt = {dt_per_frame_ps:.2f} ps/frame, span = {time_ps[-1]:.1f} ps"
)

### Check 1 — density plateau

Under anisotropic NPT the three monoclinic cell vectors relax independently against their own diagonal stress, and the volume — hence the density $\rho = m/V$ — settles onto a plateau. For naphthalene the experimental density is **1.18 g/cm³** at 295 K ([Brock & Dunitz, 1982](https://doi.org/10.1107/S0567740882008358)). The full $\rho$/temperature/cell-vector relaxation was plotted live in the warmup section above; here we read out only the tail-average density from the thinned trajectory and compare it to the experimental value.

In [ ]:
# ─── Tail-average density (no figure — the time series is shown in §7) ─────
from helpers import compute_density

density = np.array([compute_density(b)[0] for b in warmup_frames])
n_tail = max(1, len(density) // 5)
rho_tail = density[-n_tail:]
print(
    f"density (last 20%, n={n_tail}): "
    f"{rho_tail.mean():.3f} ± {rho_tail.std():.3f} g/cm³  (experimental 1.18)"
)

### Check 2 — translational diffusion $D$

A crystal's molecules vibrate about fixed lattice sites but do not *translate*; a liquid's molecules wander freely. We separate the two by tracking the molecular **centre-of-mass** mean-squared displacement and fitting its long-time slope to the 3-D Einstein relation $\mathrm{MSD} = 6 D t$.

Two subtleties make this non-trivial here, and `compute_com_msd` already handles both: under periodic boundaries the displacement is accumulated in **fractional** coordinates using each frame's own (breathing) NPT cell, and the rigid system-wide drift is subtracted every step. Without these, a perfectly pinned crystal would appear to diffuse by one–two orders of magnitude — purely an artefact of the wandering cell and thermostat-induced centre-of-mass drift. We take the helper's defaults (`subtract_system_com=True`) rather than reimplement the unwrap.

A diffusion coefficient below $\approx 10^{-7}$ cm²/s is indistinguishable from a non-diffusing crystal.

In [ ]:
# ─── COM-MSD → Einstein diffusion coefficient ──────────────────────────────
masses_ref = warmup_frames[0].atomic_masses
msd_per_mol = compute_com_msd(snap_positions, snap_cells, masses_ref, ATOMS_PER_MOL)
fit = fit_diffusion_coefficient(msd_per_mol.numpy(), time_ps[1:], fit_frac=0.5)

D_cm2_per_s = fit["D_cm2_per_s"]
fit_t = time_ps[1:][fit["fit_start_idx"]:]
fit_line = fit["slope"] * fit_t + fit["intercept"]
regime = ("crystal (D ≈ 0)" if abs(D_cm2_per_s) < 1e-7
          else "plastic-crystal-like" if abs(D_cm2_per_s) < 1e-6
          else "liquid-like")

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=_NV["dark"])
ax.plot(time_ps[1:], fit["msd_mean"], color=_NV["green"], lw=1.6, label="COM-MSD (cross-molecule mean)")
ax.axvspan(fit_t[0], fit_t[-1], color=_NV["blue"], alpha=0.15, label="Einstein fit window (last 50%)")
ax.plot(fit_t, fit_line, color=_NV["blue"], ls="--", lw=1.8,
        label=f"fit: D = {D_cm2_per_s:+.2e} cm$^2$/s")
_style_dark_ax(ax, xlabel="time (ps)", ylabel="COM-MSD (Å$^2$)",
               title=f"Translational diffusion — T = {T_WARMUP:.0f} K  →  {regime}")
ax.legend(facecolor=_NV["dark"], edgecolor=_NV["spine"], labelcolor=_NV["light"], loc="upper left")
fig.tight_layout()
plt.show()

print(f"D = {D_cm2_per_s:+.3e} cm²/s   (regime: {regime})")

### Check 3 — rotational order $S_0$

Translational arrest alone does not prove a *crystal*: a **plastic crystal** is locked on its lattice sites yet tumbles freely. We separate these with the rotational order parameter $S_0$ — the long-time tail of the second-rank ($P_2$) autocorrelation of each molecule's three principal-inertia axes (long in-plane, short in-plane, plane normal) relative to the first warmup-NPT frame. $P_2(x) = (3x^2 - 1)/2$ is sign-invariant, so the arbitrary sign of the inertia eigenvectors does not matter.

$S_0 \to 1$ means orientations are perfectly preserved (ordered crystal); intermediate $0 < S_0 < 1$ is a plastic crystal; $S_0 \to 0$ is isotropic tumbling (liquid).

In [ ]:
# ─── Rotational order parameter S0 ─────────────────────────────────────────
s0_per_mol, s0_per_mol_per_axis, acf = compute_S0_from_frames(
    warmup_frames, ATOMS_PER_MOL, ref_idx=0, tail_frac=0.2
)
S0 = float(s0_per_mol.mean())
S0_axes = s0_per_mol_per_axis.mean(axis=0)       # [3]: long / short / normal
acf_axis_mean = acf.mean(axis=1)                  # [n_frames, 3] cross-molecule mean per axis

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor=_NV["dark"])
axis_labels = ["long axis (smallest I)", "short axis (middle I)", "plane normal (largest I)"]
axis_colors = [_NV["green"], _NV["blue"], _NV["muted"]]
for k, (lbl, col) in enumerate(zip(axis_labels, axis_colors)):
    ax.plot(time_ps, acf_axis_mean[:, k], color=col, lw=1.6, label=lbl)
ax.plot(time_ps, acf.mean(axis=(1, 2)), color=_NV["light"], lw=2.2, ls="--", label=f"mean S₀ = {S0:.3f}")
ax.axhline(1.0, color=_NV["spine"], ls=":", lw=0.8)
ax.axhline(0.0, color=_NV["spine"], ls=":", lw=0.8)
ax.set_ylim(-0.15, 1.15)
_style_dark_ax(ax, xlabel="time (ps)", ylabel="P$_2$ rotational ACF",
               title=f"Rotational order — warmup NPT @ T = {T_WARMUP:.0f} K")
ax.legend(facecolor=_NV["dark"], edgecolor=_NV["spine"], labelcolor=_NV["light"], loc="lower left")
fig.tight_layout()
plt.show()

print(f"S₀ per axis (long / short / normal): {S0_axes[0]:+.3f}  {S0_axes[1]:+.3f}  {S0_axes[2]:+.3f}")
print(f"S₀ mean:                             {S0:+.3f}")

### Phase read: the warmup endpoint is a crystal

The two order parameters together classify the phase — the same table we will apply to every coexistence temperature in the melting-point screen:

| signature | $D$ | $S_0$ | phase |
|---|---|---|---|
| ordered crystal | $\approx 0$ | $\to 1$ | **solid** — translational *and* orientational order |
| plastic crystal | $\approx 0$ | $0 < S_0 < 1$ | sites fixed, molecules tumble |
| liquid | $\gg 0$ | $\to 0$ | free translation *and* rotation |

For the warmup endpoint we measured $D \approx 0$ (well below the $\approx 10^{-7}$ cm²/s crystal threshold) and $S_0 \to 1$ — the unambiguous **crystal** signature. The warm crystal is a sound solid input to the coexistence stack. A liquid- or plastic-like read here would mean lowering $T_\mathrm{warmup}$ or auditing the model and equilibration before going further.

## The liquid half

Solid–liquid coexistence needs a **liquid slab** sitting against the crystal in one cell. The
liquid must be a genuinely disordered configuration at roughly the same density as the solid — if
the two halves differ wildly in density the interface relaxes by bulk flow rather than by melting
or freezing, and the coexistence signal is lost.

There are two ways to obtain that disordered half:

- **Melt it dynamically.** Take the warm crystal, reseed velocities far above the melting point,
  and run MD until it loses order. This works, but it is slow (you pay for the melting trajectory),
  the final density depends on the ensemble you chose, and a too-fast melt can leave trapped voids
  or local overlaps that the downstream minimiser then has to fight.
- **Pack it geometrically.** Place the molecules directly, with a hard minimum-separation
  constraint, at a density you choose. This is what **Packmol**
  ([Martínez et al., 2009](https://doi.org/10.1002/jcc.21224)) does: it solves a constrained
  optimisation that drops `N` copies of a molecule into a region such that no two atoms come closer
  than a tolerance — including periodic images and any fixed obstacle. It is **not** molecular
  dynamics; it is seconds of geometry, fully reproducible from a seed, and it gives us direct
  control over count, region, and minimum contact.

We use Packmol. It is the same tool the methodology reference builds its coexistence cells with
([Schmidt, Van der Spoel & Walz, 2023](https://doi.org/10.1021/acsphyschemau.2c00045)), and it
hands the subsequent FIRE2 + NVT pre-equilibration a clean, overlap-free starting point.

> **Prerequisite.** The packing is driven by the `packmol` Python package (shipped in the
> container's `requirements.txt`), which bundles a static Packmol binary; `helpers.packmol`
> locates it via `packmol.cli.get_binary_path()`. No system install or `$PATH` entry is needed.

### Building it with `helpers.packmol`

Two helpers do the work:

- **`extract_single_molecule(unit_cell)`** pulls one molecule out of the CIF unit cell. It walks
  covalent connectivity (largest connected component), stitches any bond that wrapped across the
  cell boundary back together via the minimum-image convention, and returns a clean, periodic-free
  C<sub>10</sub>H<sub>8</sub> template — the building block Packmol will replicate.
- **`pack_liquid_box(molecule, n_molecules, target_density, model_cutoff, tolerance, nloop)`** sizes
  a cubic periodic box to the target density and packs `n_molecules` copies into it, keeping every
  atom pair (including periodic images) at least `tolerance` Å apart. The box side is automatically
  grown to at least twice the model cutoff so the minimum-image convention is valid for Orb-v3's
  6 Å cutoff.

Here we pack **`N_MOL` = 200** molecules — the same count as the crystal half — so the coexistence
cell built in the next section is symmetric. We use Packmol's stock minimum separation
(`tolerance = 2.0` Å); at organic-liquid densities the pack converges in seconds.

This cell builds a self-contained **cubic preview** of the liquid half to show what the packing
produces. The next section reuses the very same monomer and density, but confines the liquid to the
parallelepiped above the crystal with the warm crystal held fixed as an obstacle — so the two halves
share one cell and Packmol's tolerance prevents clashes across the interface at construction time.

In [ ]:
# ─── Extract one naphthalene from the CIF (the packing template) ────────────
from helpers import extract_single_molecule

monomer = extract_single_molecule(unit_cell)
print(f"Monomer: {monomer.get_chemical_formula()}  ({len(monomer)} atoms)")
print(f"Packing {N_MOL} copies (same count as the crystal half)")

In [ ]:
# ─── Pack a disordered liquid preview (Packmol; seconds, CPU) ───────────────
from helpers import AMU_OVER_A3_TO_G_CM3, pack_liquid_box

ORB_CUTOFF = 6.0          # Å — Orb-v3 neighbour cutoff (box must exceed 2× this)
LIQUID_DENSITY = 1.18     # g/cm³ — match the crystal density for the preview

liquid_atoms, n_packed = pack_liquid_box(
    monomer,
    n_molecules=N_MOL,
    target_density=LIQUID_DENSITY,
    model_cutoff=ORB_CUTOFF,
    tolerance=2.0,        # Packmol stock minimum inter-atom separation (Å)
    nloop=50,             # Packmol stock GENCAN iteration cap
)

box_side = liquid_atoms.cell.lengths()[0]
rho = liquid_atoms.get_masses().sum() / liquid_atoms.get_volume() * AMU_OVER_A3_TO_G_CM3
print(f"Packed {n_packed} molecules ({len(liquid_atoms)} atoms) "
      f"into a {box_side:.1f} Å cubic box at {rho:.3f} g/cm³")

In [ ]:
# ─── Crystal (ordered) vs liquid preview (disordered) ───────────────────────
from helpers import display_widgets_row

display_widgets_row(
    [
        (f"crystal — {N_MOL} molecules (ordered)", supercell),
        (f"liquid preview — {n_packed} molecules (Packmol)", liquid_atoms),
    ],
    width="380px",
    height="360px",
    show_cell=True,
)

## Stacking the coexistence cell

Solid–liquid coexistence ([Schmidt, Van der Spoel & Walz, 2023](https://doi.org/10.1021/acsphyschemau.2c00045))
puts the crystal and the liquid in **one** simulation cell, sharing a flat interface. As the
production NPT runs, the interface advances: below the melting point the liquid freezes onto the
crystal, above it the crystal melts. We bracket $T_\mathrm{m}$ from which way the interface moves at
each temperature — no superheating, no nucleation barrier to cross.

**Which axis do we stack along?** The interface plane must be a true lattice plane with its normal
*parallel* to the stacking direction, or anisotropic NPT will shear it. For monoclinic $P2_1/a$
naphthalene ($\beta \approx 124°$) only the **unique axis $b$** satisfies $\vec b \parallel \vec b^*$
(the direct lattice vector is perpendicular to its own face). The $a$ and $c$ vectors are tilted off
their face normals by $\beta - 90° \approx 34°$, so stacking along them would give an oblique
interface that bends under the barostat. We stack along $b$ = `cell[1, :]` and double it, giving the
(010) interface.

**Unwrapping molecules first.** Both halves arrive periodic-wrapped: any molecule straddling a cell
face has its atoms mapped to the opposite side. Concatenating and then doubling the cell would leave
those molecules *torn* — atoms separated by a full $|b|$, read by the model as dissociated fragments.
So each half is unwrapped in **its own cell** (minimum-image, relative to each molecule's first atom)
and folded back by molecular centre-of-mass *before* concatenation, and the stacked configuration is
re-stitched once more in the doubled cell.

**No vacuum gap — Packmol enforces the interface.** Because we build the liquid half with **Packmol**
([Martínez et al., 2009](https://doi.org/10.1002/jcc.21224)) holding the crystal as a *fixed
obstacle*, Packmol's minimum-separation `tolerance` already keeps every placed liquid atom clear of
every crystal atom (including periodic images) *at construction time*. The stacked cell is therefore
exactly $(a,\, 2b,\, c)$ with no inserted vacuum — the tolerance check replaces the gap that a
melt-then-stack workflow would need. We confirm the result with a periodic nearest-neighbour distance
check before any dynamics.

### One call builds the stack

`helpers.build_packmol_slc_stack(crystal_batch, monomer, n_melt_molecules)` does the whole
construction in a single Packmol invocation:

1. **Unwrap + COM-wrap** the crystal molecules in the crystal cell (stitching any periodic splits from
   the warmup endpoint).
2. **Build the stacked cell** $(a,\, 2b,\, c)$ and a coordinate shift that places its orthorhombic
   bounding box at the world origin (Packmol's periodic box is axis-aligned).
3. **Carve the upper-$b$ parallelepiped** for the liquid with six `above`/`below plane` constraints, so
   the packed molecules land in the top half against the crystal.
4. **Run Packmol once** with the crystal as a fixed obstacle and `pbc` over the bounding box, so the
   `tolerance` check sees the crystal atoms *and* their periodic images — no cross-interface clashes.
5. **Re-stitch** any molecule the bounding-box wrap split, and return a single-graph `Batch` with the
   crystal atoms first and the Packmol-placed liquid atoms after.

This is the same routine the consolidated SLC driver uses, so the notebook and the production run build
geometrically identical stacks. It is plumbing-heavy (bounding-box shifts, plane normals, periodic
re-stitching), so unlike the model wrapper we **import** it rather than reproduce it inline — the
toolkit-teaching content (multi-graph batching, per-graph temperatures) comes next and is where the
notebook spends its detail.

In [ ]:
# ─── Build the SLC stack: crystal (fixed) + Packmol liquid half ─────────────
from helpers import build_packmol_slc_stack, extract_single_molecule

# The crystal half is the warmup-NPT endpoint. The 50 ps warmup NPT is never
# run live (compute mode runs only the cheap FIRE + 1 ps NVT), so the
# equilibrated crystal is always materialised from the shipped cache.
warmup_npt_path = cfg.cached_extxyz("warmup_npt")
if warmup_npt_path is None:
    raise FileNotFoundError(
        "warmup_npt trajectory not in cache — stage the cached run first "
        "(tools/stage_cached_run.py)."
    )
crystal_atoms = ase_read(str(warmup_npt_path), index=-1)   # last (equilibrated) frame
crystal_batch = ase_frames_to_batches([crystal_atoms], device=DEVICE)[0]

monomer = extract_single_molecule(unit_cell)               # one C10H8 (PBC-free)
n_mol_crystal = crystal_batch.num_nodes // len(monomer)

slc_batch = build_packmol_slc_stack(
    crystal_batch, monomer, n_mol_crystal,
    device=str(DEVICE), tolerance=2.0, nloop=20,
)
print(f"Monomer: {monomer.get_chemical_formula()} ({len(monomer)} atoms)")
print(f"SLC stack: {slc_batch.num_nodes} atoms "
      f"({n_mol_crystal} crystal + {n_mol_crystal} liquid molecules)")

In [ ]:
# ─── Clash check: no cross-interface overlaps before dynamics ───────────────
from helpers import min_pbc_distance

slc_cl = slc_batch.cell.squeeze().norm(dim=-1)
min_dist = min_pbc_distance(slc_batch.positions, slc_batch.cell.squeeze())
print(f"Cell lengths (a, b, c): {[f'{l:.2f}' for l in slc_cl.tolist()]} Å  (b doubled)")
print(f"Min periodic pairwise distance: {min_dist:.2f} Å  (must exceed 0.5 Å)")
assert min_dist > 0.5, f"Clash at the interface: {min_dist:.2f} Å — check the pack tolerance"

In [ ]:
# ─── The stacked coexistence cell (crystal half + liquid half) ──────────────
from helpers import batch_to_ase, display_widgets_row

display_widgets_row(
    [
        (f"crystal half — {n_mol_crystal} molecules (ordered)", supercell),
        (f"SLC stack — {slc_batch.num_nodes} atoms, (010) interface", batch_to_ase(slc_batch)),
    ],
    width="380px",
    height="380px",
    show_cell=True,
)

## One batch, four temperatures

We need the coexistence run at several temperatures to bracket $T_\mathrm{m}$. Rather than four
separate simulations, we run them as **one multi-graph `Batch`** — four identical copies of the stacked
cell, one per target temperature in `TEMPS` = {200, 300, 400, 500} K — and let each *graph* carry its
**own integrator temperature**.

The Toolkit's integrators accept temperature as a tensor of shape `[M]` (one value per graph) and
broadcast it to every atom through `batch.batch_idx`. So a single `NVTLangevin` (and, in the next
section, a single `NPT`) drives all four systems at once: one GPU launch, four temperatures, near-linear
throughput — the same batching win we measured earlier, now doing real physics. This is the pattern
that makes a GPU melting-point sweep practical, and it carries straight through to the production NPT.

In [ ]:
# ─── Per-temperature setup for the multi-graph sweep ────────────────────────
n_slc_nvt = int(SLC_NVT_PS * 1000 / DT)                 # NVT steps per T (1 ps -> 2000)
temps_tensor = torch.tensor([float(T) for T in TEMPS], device=DEVICE)
t_labels = [f"T={int(T)}K" for T in TEMPS]              # legible per-graph CSV/stdout tags
slc_fire_stem = "slc_fire"                              # FIRE is single-graph (shared)
slc_nvt_stems = {int(T): f"slc_nvt_t{int(T)}" for T in TEMPS}
print(f"Sweep: {len(TEMPS)} graphs at {t_labels}; NVT {n_slc_nvt} steps/T (DT={DT} fs)")

### Pre-equilibrating the interface

Before the production NPT, two short stages settle the freshly-built interface
([Schmidt et al., 2023](https://doi.org/10.1021/acsphyschemau.2c00045)):

1. **FIRE2 minimisation** on the single stacked geometry — drains the residual elastic strain left
   where the packed liquid meets the crystal, so the production barostat does not see a force spike at
   step 0.
2. **A short NVT** (1 ps) at each target temperature — brings both halves to their shared temperature
   and lets the interface relax thermally before the cell is allowed to move. This is where the
   multi-graph batch comes in: we clone the minimised geometry once per temperature and thermalise all
   four in parallel.

Both stages are cheap, so **they run live** when `RESULT_SOURCE = "compute"`. The long production NPT
(300 ps per temperature) is always replayed from the shipped cache — we never wait hours for it in the
notebook. The per-temperature detail of the live NVT is captured in its CSV log; the progress bar
tracks the shared step count across all four graphs.

In [ ]:
# ─── Stage A: FIRE2 minimise the stacked geometry (single graph) ────────────
from nvalchemi.dynamics import ConvergenceHook
from nvalchemi.dynamics.hooks import LoggingHook, SnapshotHook
from nvalchemi.dynamics.optimizers import FIRE2

from helpers import (
    DYNAMICS_SCALARS, NotebookProgress, fresh_zarr_sink,
    make_progress_writer, make_safety_hooks,
)

if USE_SAVED:
    fire_path = cfg.cached_extxyz(slc_fire_stem)
    if fire_path is None:
        print(f"[saved] {slc_fire_stem} not staged yet — using the freshly stacked geometry.")
    else:
        slc_batch = ase_frames_to_batches([ase_read(str(fire_path), index=-1)], device=DEVICE)[0]
        print(f"[saved] loaded FIRE2 endpoint from {fire_path.name}")
else:
    progress = NotebookProgress(title="SLC FIRE2 minimisation",
                                total=LIVE_FIRE_MAX_STEPS, unit="steps",
                                average_label="steps")
    fire_zarr = fresh_zarr_sink(LOG_DIR / f"{slc_fire_stem}.zarr",
                                capacity=LIVE_FIRE_MAX_STEPS // SNAPSHOT_EVERY + 10)
    fire_csv = LoggingHook(backend="csv", custom_scalars=DYNAMICS_SCALARS,
                           log_path=str(LOG_DIR / f"{slc_fire_stem}.csv"), frequency=LOG_EVERY)
    fire_out = LoggingHook(backend="custom", writer_fn=make_progress_writer(progress),
                           custom_scalars=DYNAMICS_SCALARS, frequency=LOG_EVERY)
    fire = FIRE2(model=model, dt=0.005, tmax=0.01, n_steps=LIVE_FIRE_MAX_STEPS,
                 convergence_hook=ConvergenceHook.from_fmax(threshold=FMAX))
    for h in [*make_safety_hooks(model, track_stress=False),
              SnapshotHook(sink=fire_zarr, frequency=SNAPSHOT_EVERY), fire_csv, fire_out]:
        fire.register_hook(h)
    with fire_csv, fire_out:
        slc_batch = fire.run(slc_batch)
    progress.update(done=LIVE_FIRE_MAX_STEPS, message="minimised")
    print(f"FIRE2 done: {slc_batch.num_nodes} atoms minimised")

In [ ]:
# ─── Stage B: short NVT @ each target T (multi-graph) ───────────────────────
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import initialize_velocities
from nvalchemi.dynamics.integrators.nvt_langevin import NVTLangevin

from helpers import (
    DYNAMICS_SCALARS, NotebookProgress, fresh_zarr_sink,
    make_progress_writer, make_safety_hooks,
)


def _clone_multi_graph(single, n_copies):
    """Clone a single-graph Batch into n_copies independent graphs (zero velocities)."""
    n = single.num_nodes
    datas = []
    for _ in range(n_copies):
        d = AtomicData(
            positions=single.positions.clone(),
            atomic_numbers=single.atomic_numbers.clone(),
            velocities=torch.zeros_like(single.positions),
            forces=torch.zeros(n, 3, device=DEVICE),
            energy=torch.zeros(1, 1, device=DEVICE),
            stress=torch.zeros(1, 3, 3, device=DEVICE),
            cell=single.cell.squeeze().clone().unsqueeze(0),
            pbc=torch.tensor([[True, True, True]], device=DEVICE),
        )
        d.charge = torch.zeros(1, 1, device=DEVICE)
        datas.append(d)
    return Batch.from_data_list(datas, device=DEVICE)


if USE_SAVED:
    # §11 reads its own per-T production NPT trajectories, so we only need a
    # multi-graph batch with num_graphs == len(TEMPS) to carry the geometry
    # forward — clone the stacked cell once per temperature (zero velocities).
    slc_multi_batch = _clone_multi_graph(slc_batch, len(TEMPS))
    print("[saved] pre-equilibration shown from cache; §11 reads the per-T production NPT trajectories.")
else:
    slc_multi_batch = _clone_multi_graph(slc_batch, len(TEMPS))
    initialize_velocities(
        slc_multi_batch.velocities, slc_multi_batch.atomic_masses,
        temperature=temps_tensor, batch_idx=slc_multi_batch.batch_idx,
        random_seed=42, remove_com=True, rescale=True,
    )
    nvt_progress = NotebookProgress(title="SLC NVT — 4 temperatures",
                                    total=n_slc_nvt, unit="steps",
                                    average_label="steps")
    nvt_zarr = fresh_zarr_sink(LOG_DIR / "slc_nvt.zarr",
                               capacity=len(TEMPS) * (n_slc_nvt // SNAPSHOT_EVERY) + 10)
    nvt_csv = LoggingHook(backend="csv", custom_scalars=DYNAMICS_SCALARS,
                          log_path=str(LOG_DIR / "slc_nvt.csv"), frequency=LOG_EVERY)
    nvt_out = LoggingHook(backend="custom", writer_fn=make_progress_writer(nvt_progress),
                          custom_scalars=DYNAMICS_SCALARS, frequency=LOG_EVERY)
    nvt = NVTLangevin(model=model, dt=DT, temperature=temps_tensor,
                      friction=FRICTION, n_steps=n_slc_nvt)
    for h in [*make_safety_hooks(model),
              SnapshotHook(sink=nvt_zarr, frequency=SNAPSHOT_EVERY), nvt_csv, nvt_out]:
        nvt.register_hook(h)
    with nvt_csv, nvt_out:
        slc_multi_batch = nvt.run(slc_multi_batch)
    nvt_progress.update(done=n_slc_nvt, message="thermalised")

print(f"SLC pre-equilibration complete; {slc_multi_batch.num_graphs} graphs ready for production NPT.")

## Production coexistence: anisotropic NPT, shown from the cache

The pre-equilibrated stack is now run under **constant pressure** so the box can
respond to melting or freezing at the interface. All four temperatures are
advanced together as **one multi-graph `Batch`** — the §10 batching showcase —
each graph carrying its own target temperature, so a single `NPT(...).run(...)`
broadcasts the per-graph temperature through `batch.batch_idx`.

The pressure coupling is **anisotropic** and hydrostatic. Naphthalene is
monoclinic and the two-phase stack has a real anisotropy: the interface-normal
axis (the monoclinic unique axis **b**, along which we stacked) must be free to
lengthen or shorten as the solid/liquid fraction shifts, while the in-plane axes
stay pinned by the crystal lattice. A scalar/isotropic barostat would force
uniform dilation and bias the apparent melting point. We therefore pass a
**rank-2 pressure tensor** `torch.tensor([[P, P, P]])` (shape `[1, 3]`, broadcast
per graph) with `pressure_coupling="anisotropic"` — 1 atm hydrostatic on each
cell diagonal. Toolkit units are **eV/Å³** for pressure and **fs⁻¹** for
friction, and the stress the model returns is **tension-positive Cauchy** (no
sign flip). The coupling times are the canonical τ_P = `BAROSTAT_TIME` and
τ_T = `THERMOSTAT_TIME`, matching the warmup NPT so the cell dynamics stay
consistent across the chain.

> **This stage is shown from the cache — it is never run live.** Four
> temperatures × 300 ps × 0.5 fs ≈ 2.4 × 10⁶ integration steps over a 7200-atom
> stack is several GPU-hours; it does not belong in an interactive tutorial.
> Even with `RESULT_SOURCE="compute"` the cheap pre-equilibration runs live, but
> this production NPT is always replayed from the shipped trajectories and logs.
> The production driver in the dev sources reproduces it end-to-end if you want
> to regenerate the run yourself.

To read the melting point we bracket it with the **two endpoints that fall
cleanly on either side of the transition** — **200 K** (well below) and **500 K**
(well above). The intermediate 300 K and 400 K cells are part of the batch but
sit near the transition where the two-point assignment is ambiguous, so we hold
them back and screen with the unambiguous endpoints. The cached time-series
below shows temperature, density, pressure, and energy settling into per-graph
steady states for those two cells.

In [ ]:
# ─── Endpoint time series, rendered live from the cached NPT logs ───────────
from helpers import plot_slc_stage

BRACKET_TEMPS = (200, 500)  # crystal / melt endpoints of the coexistence screen

csv_by_temp = {T: cfg.cached_log_csv(f"slc_npt_t{T}") for T in BRACKET_TEMPS}
if any(p is None for p in csv_by_temp.values()):
    missing = [T for T, p in csv_by_temp.items() if p is None]
    print(f"[cache pending] production-NPT logs for {missing} K not found under "
          f"{cfg.CACHE_CSV_DIR}; this stage is cached-only and is not run live.")
else:
    fig = plot_slc_stage({T: str(p) for T, p in csv_by_temp.items()}, dt=DT)
    display(fig)
    plt.close(fig)

In [ ]:
# ─── Load the cached endpoint production trajectories ───────────────────────
# Cached-only (see above). Each temperature was written as its own thinned
# extxyz; load the two endpoints into per-T single-graph Batch lists ready for
# the analysis helpers. (If you regenerated the run as one multi-graph batch you
# would instead de-interleave with batches[i::len(TEMPS)].)
results = {}
for T in BRACKET_TEMPS:
    path = cfg.cached_extxyz(f"slc_npt_t{T}")
    if path is None:
        raise FileNotFoundError(
            f"cached production trajectory 'slc_npt_t{T}.extxyz' not found under "
            f"{cfg.CACHE_TRAJ_DIR}. The 300 ps NPT is never run live — it must be "
            f"present in the shipped cache."
        )
    frames = ase_read(str(path), index=":")
    results[T] = ase_frames_to_batches(frames)
    print(f"  T = {T:>3} K  →  {len(frames)} frames")

## Reading the melting point off two signatures

[Schmidt, Van der Spoel & Walz (2023)](https://doi.org/10.1021/acsphyschemau.2c00045)
screen melting points by watching how a solid–liquid coexistence cell evolves at
each temperature, using two order parameters per phase:

- **Rotational order S₀** — the tail value of the second-rank (P₂) rotational
  autocorrelation of each molecule's principal axes. `S₀ → 1` means orientations
  are frozen (crystalline); `S₀ → 0` means free rotation (liquid).
- **Translational diffusion D** — from the long-time slope of the molecular
  centre-of-mass mean-squared displacement, `MSD = 6 D t` (the Einstein
  relation). `D ≈ 0` is a solid; `D ≫ 0` is a liquid.

Together they classify each phase:

| signature | crystal | plastic crystal | liquid |
|---|---|---|---|
| **D** | ≈ 0 | ≈ 0 | ≫ 0 |
| **S₀** | → 1 | 0 < S₀ < 1 | → 0 |

We apply the classifier **separately to each half** of the coexistence stack —
the crystal half is the first `n_half` atoms (`atom_slice=slice(0, n_half)`), the
liquid half is the rest (`slice(n_half, None)`). The crystal half is the
diagnostic: below the melting point it stays ordered and pinned (`S₀ → 1`,
`D ≈ 0`); above it, the solid front recedes and the crystal half disorders
(`S₀` collapses, `D` jumps). **Schmidt et al.'s rule** then places the melting
point at the **midpoint between the highest temperature that stays solid and the
lowest that melts** — here the centre of the [200, 500] K endpoint bracket.

> **A subtlety in the MSD.** Under periodic boundaries and an anisotropic
> barostat the cell is deforming and individual atoms wrap across the box, so a
> naive displacement is meaningless. `compute_com_msd` handles both: it collapses
> each molecule to a mass-weighted, PBC-unwrapped centre of mass, accumulates the
> *fractional* displacement frame-to-frame (so affine cell deformation
> contributes zero), and subtracts the system-wide drift. A molecule pinned to
> its lattice site then reads `MSD ≈ 0` even while the box breathes.

In [ ]:
# ─── Per-endpoint observables: density, per-half S₀, crystal-half COM-MSD → D ─
from helpers import compute_S0_from_frames, compute_com_msd, compute_density
from helpers.diffusion import fit_diffusion_coefficient

# Stack is crystal-half-first, liquid-half-second; split at the midpoint. Atoms
# are written in contiguous per-molecule blocks (compute_com_msd /
# compute_S0_from_frames require this).
n_half = next(iter(results.values()))[0].num_nodes // 2

# Cached frames are written every 10000 steps → 5 ps apart (NOT SNAPSHOT_EVERY).
dt_per_frame_ps = 10000 * DT / 1000.0

per_T = {}
for T in BRACKET_TEMPS:
    frames = results[T]
    masses = frames[0].atomic_masses.cpu()
    pos = [b.positions for b in frames]
    cells = [b.cell.squeeze() for b in frames]

    # Per-half rotational order S₀ (tail-averaged P₂ ACF).
    S0_c, _, _ = compute_S0_from_frames(frames, ATOMS_PER_MOL, atom_slice=slice(0, n_half))
    S0_m, _, _ = compute_S0_from_frames(frames, ATOMS_PER_MOL, atom_slice=slice(n_half, None))

    # Crystal-half COM-MSD → D (Einstein); slice the crystal half's atoms/masses.
    msd_c = compute_com_msd(
        [p[:n_half] for p in pos], cells, masses[:n_half], ATOMS_PER_MOL
    ).numpy()
    time_ps = np.arange(1, msd_c.shape[0] + 1) * dt_per_frame_ps
    D_c = fit_diffusion_coefficient(msd_c, time_ps, fit_frac=0.5)["D_cm2_per_s"]

    rho = np.array([compute_density(b)[0] for b in frames])
    per_T[T] = {
        "rho": float(rho[len(rho) // 2:].mean()),  # steady-state (last half) mean
        "S0_c": float(S0_c.mean()),
        "S0_m": float(S0_m.mean()),
        "D_c_cm2_s": float(D_c),
        "phase": "crystal" if T == min(BRACKET_TEMPS) else "melt",
    }

print(f"{'T (K)':>6} {'phase':>8} {'ρ (g/cm³)':>11} {'S0 cryst':>9} "
      f"{'S0 liq':>8} {'D cryst (cm²/s)':>17}")
print("-" * 64)
for T in BRACKET_TEMPS:
    a = per_T[T]
    print(f"{T:>6} {a['phase']:>8} {a['rho']:>11.3f} {a['S0_c']:>9.3f} "
          f"{a['S0_m']:>8.3f} {a['D_c_cm2_s']:>17.2e}")

# Schmidt bracket-midpoint rule on the two clean endpoints.
tm_lo, tm_hi = min(BRACKET_TEMPS), max(BRACKET_TEMPS)  # solid / melt bounds
tm_mid = (tm_lo + tm_hi) / 2
print(f"\nCoexistence bracket [{tm_lo}, {tm_hi}] K  →  midpoint Tₘ ≈ {tm_mid:.0f} K"
      f"   (experimental {int(TM_EXP)} K)")

### Watch it melt: the 500 K coexistence cell

The table is two numbers per half, but melting is a spatial process — it starts
at the interface and eats into the solid. The live animation below replays the
500 K production trajectory. The crystal half (lower) loses its order and the
solid front recedes into the liquid as the cell evolves, while at 200 K the same
crystal half stays intact for the full 300 ps. Looking along the crystal **a**
axis, the stacking axis **b** runs vertically, so the two phases sit one above
the other and the interface is the horizontal seam between them.

In [ ]:
# ─── Live OVITO melting animation of the 500 K coexistence cell ─────────────
from helpers import display_trajectory_animation

_path_500 = cfg.cached_extxyz("slc_npt_t500")
if _path_500 is None:
    print("[cache pending] slc_npt_t500.extxyz not found; melting animation is cached-only.")
else:
    display_trajectory_animation(
        ase_read(str(_path_500), index=":"),
        label="naphthalene SLC · melting at 500 K",
        target_frames=ANIM_TARGET_FRAMES,
    )

In [ ]:
# ─── Melting-point bracket figure: density, crystal-half S₀, crystal-half D ──
nv_green, nv_blue, nv_amber = "#76B900", "#00A3E0", "#F5A623"
dark, light, grid = "#000000", "#F3F5F7", "#2F3A44"

T_arr = np.array(BRACKET_TEMPS, dtype=float)
rho = np.array([per_T[T]["rho"] for T in BRACKET_TEMPS])
S0_c = np.array([per_T[T]["S0_c"] for T in BRACKET_TEMPS])
D_c = np.abs(np.array([per_T[T]["D_c_cm2_s"] for T in BRACKET_TEMPS]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), facecolor=dark)
panels = [
    (axes[0], rho, "Density (g/cm³)", "Steady-state density", nv_green, "o-", False),
    (axes[1], S0_c, "S₀ (crystal half)", "Rotational order", nv_blue, "s-", False),
    (axes[2], D_c, "|D| crystal half (cm²/s)", "Translational diffusion", nv_amber, "^-", True),
]
for ax, y, ylabel, title, color, style, logy in panels:
    ax.set_facecolor(dark)
    ax.plot(T_arr, y, style, color=color, markersize=10, lw=2.4)
    ax.axvspan(tm_lo, tm_hi, color=nv_green, alpha=0.10)
    ax.axvline(TM_EXP, color=light, ls="--", lw=1.4, label=f"exp {int(TM_EXP)} K")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("Temperature (K)", color=light, fontsize=12)
    ax.set_ylabel(ylabel, color=light, fontsize=12)
    ax.set_title(title, color=light, fontsize=12, pad=8)
    ax.set_xlim(tm_lo - 30, tm_hi + 30)
    ax.set_xticks(list(T_arr))
    ax.tick_params(colors=light, labelsize=11)
    for s in ax.spines.values():
        s.set_color("#4B5563")
    ax.grid(True, color=grid, lw=0.8, alpha=0.7)
    ax.legend(facecolor=dark, edgecolor="#4B5563", labelcolor=light, fontsize=9, loc="best")

fig.suptitle(
    f"Naphthalene melting-point screen — coarse bracket {tm_lo}–{tm_hi} K "
    f"(exp {int(TM_EXP)} K)",
    color=light, fontsize=15, y=0.99,
)
fig.tight_layout(rect=(0, 0, 1, 0.93))
out = Path("assets/images/tm_bracket.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=320, bbox_inches="tight", facecolor=fig.get_facecolor())
display(fig)
plt.close(fig)

## The verdict — a coarse coexistence bracket

The two endpoints fall cleanly on either side of the transition: at **200 K** the
crystal half stays ordered and immobile (`S₀ → 1`, `D ≈ 0`) and the cell is dense
— it is a crystal; at **500 K** the crystal half has disordered (`S₀` collapsed,
`D` jumped by orders of magnitude) and the cell has expanded — it is a melt. By
the [Schmidt et al. (2023)](https://doi.org/10.1021/acsphyschemau.2c00045)
midpoint rule this places the melting point at the **centre of the [200, 500] K
bracket — ≈ 350 K, essentially on the experimental naphthalene value of 353 K**.

This is a **coarse screening estimate, not a converged melting point**. A
two-point bracket pins Tₘ only to within a ±150 K half-range; the close agreement
of the midpoint with 353 K is encouraging but should not be read as that
precision. Sharpening it would take a finer temperature scan around the bracket
or a free-energy method (e.g. interface pinning or thermodynamic integration)
rather than a watch-it-coexist screen. We take up these limitations — finite
size, the single coexistence geometry, and model accuracy — in the closing
section.

## Interpreting the melting-point screen

A coexistence simulation does not measure $T_\mathrm{m}$ directly. At each temperature it answers a single, sharp question: held at constant pressure with a solid and a liquid sharing one interface, **does the interface advance toward the solid or toward the liquid?** Below the melting point the ordered phase wins and the interface freezes inward; above it the disordered phase wins and the crystal melts. The temperature where the interface neither grows nor recedes *is* $T_\mathrm{m}$.

Sweeping $T \in \{200, 300, 400, 500\}\ \mathrm{K}$ as one multi-graph batch turns that yes/no test into a **bracket**: the highest temperature whose C<sub>10</sub>H<sub>8</sub> interface still crystallises sets the lower bound, and the lowest temperature whose interface melts sets the upper bound. Reading the per-temperature phase classifier from §11 — diffusion $D$ together with the rotational order parameter $S_0$ — the coexistence cell is **crystalline at 200 K** (rotational order high, diffusion $\approx 0$) and **molten at 500 K** (diffusion jumps to liquid scale, order collapses). By the Schmidt midpoint rule this places the melting point at the centre of the coarse $[200, 500]\ \mathrm{K}$ bracket, $\approx 350\ \mathrm{K}$ — essentially the experimental $T_\mathrm{m}^{\mathrm{exp}} = 353.2\ \mathrm{K}$ for naphthalene (NIST WebBook; see References). For a coarse two-point bracket this is good agreement.

A bracket that straddles 353 K means the Orb-v3 (OMol) potential reproduces the cohesive/entropic balance of this molecular crystal well enough to place its melting point without any compound-specific fitting — encouraging for a foundation potential applied to a class of small aromatic molecules it was not trained to melt. A bracket that sits systematically high or low would flag a model bias (typically over- or under-binding of the crystal, or an under-described libration spectrum) rather than a bug in the workflow, and would tell you to widen the validation set before trusting the ranking.

That last point is the screening payoff. The same pipeline — equilibrate, build a coexistence cell, sweep temperatures in one GPU batch — runs at machine-learned-potential speed rather than DFT cost, so it can **rank a series of candidate OLED molecules by predicted $T_\mathrm{m}$** in the time a single ab-initio coexistence run would take. Melting point governs thermal stability and the processing window of an organic light-emitting diode, so an inexpensive, batched $T_\mathrm{m}$ screen is a practical first filter that routes only the most promising molecules toward expensive experiment or higher-fidelity simulation.

A few implementation choices made that screen trustworthy and are worth carrying to a new compound: the $D + S_0$ classifier resolves cases either signal alone leaves ambiguous (a plastic crystal has $D \approx 0$ yet $0 < S_0 < 1$); **anisotropic** NPT is mandatory, not merely preferable, because an isotropic barostat couples the solid-pinned lateral axes to the interface-normal axis that must move as the phase fraction shifts, biasing $T_\mathrm{m}$ by tens of kelvins; and per-half molecule unwrapping (and Packmol's minimum-separation tolerance at the interface) keep the initial FIRE relaxation from diverging.

## Some limitations of this tutorial — and how model/data selection can address them

This notebook demonstrates a **fast, batched melting-point screen**, not a converged thermodynamic measurement. The limits below define which downstream questions need additional system size, sampling, or methods.

- **A coarse two-point bracket.** The melting point is read from a 200 K (solid) / 500 K (liquid) coexistence bracket, so the estimate carries a ±150 K half-range; a finer temperature grid near the transition, or an explicit free-energy method (thermodynamic integration, interface pinning), sharpens it to a quantitative $T_\mathrm{m}$.
- **Finite size.** The screen uses a 200-molecule supercell (7200 atoms across the coexistence cell). A single interface in a small cell carries large interfacial and finite-size corrections; production $T_\mathrm{m}$ estimates use substantially larger systems — the reference SLC study runs $\geq 1000$ molecules per compound ([Schmidt, Van der Spoel & Walz, 2023](https://doi.org/10.1021/acsphyschemau.2c00045)). Increasing `SUPERCELL` reduces this bias at linear GPU cost.
- **One composition, one polymorph, one interface.** The procedure is validated here on a single compound, the experimental P2₁/a naphthalene crystal, melted across a single (010) interface. Benchmarking a potential for OLED-relevant chemistry needs a chemistry-spanning test set, and a polymorphic compound needs each polymorph screened separately.
- **Interaction range and dispersion.** Naphthalene is a weak-dipole molecule, so the Orb-v3 (OMol) neighbour cutoff is workable and OMol's training already includes dispersion-inclusive references — no explicit D3 term is added. For polar or hydrogen-bonding molecular crystals, long-range electrostatics and the dispersion convention become first-order effects on the predicted density and therefore on $T_\mathrm{m}$, and should be checked against experiment or higher-level theory before trusting a ranking.
- **Thinned-trajectory analysis.** The shipped diagnostics read decimated trajectories (snapshots spaced far apart relative to the integration step), which is ample for phase classification but blurs short-time dynamics; an $S_0$ relaxation time or a high-resolution $D$ fit would need a denser trajectory than the cache stores.

## Let's stay in touch

Deepest appreciation to everyone involved — the <span style="color:#76b900; font-weight:600;">NVIDIA ALCHEMI</span> team, Universal Display Corporation, and the OVITO developers — and especially to Wen Jie Ong, Ryan Reese, Roman Zubatyuk, Sepideh Khajehei, and the OpenHackathon team for the support, feedback, and collaboration that shaped this tutorial.

For questions, feedback, follow-up discussion, or ideas for extending the workflow, reach out to [Nikita Fedik](mailto:nfedik@nvidia.com) or [Justin Smith](mailto:jusmith@nvidia.com). We would love to hear from you and discuss how **<span style="color:#76b900; font-weight:600;">NVIDIA ALCHEMI</span>** could help accelerate your workflow.

🔗 **ALCHEMI resources:** [ALCHEMI hub](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) · [Toolkit GitHub](https://github.com/NVIDIA/nvalchemi-toolkit) · [Toolkit-Ops GitHub](https://github.com/NVIDIA/nvalchemi-toolkit-ops)

📰 **ALCHEMI blogs:** [ALCHEMI discovery blog](https://developer.nvidia.com/blog/revolutionizing-ai-driven-material-discovery-using-nvidia-alchemi/) · [Toolkit intro blog](https://developer.nvidia.com/blog/building-custom-atomistic-simulation-workflows-for-chemistry-and-materials-science-with-nvidia-alchemi-toolkit/) · [Toolkit-Ops blog](https://developer.nvidia.com/blog/accelerating-ai-powered-chemistry-and-materials-science-simulations-with-nvidia-alchemi-toolkit-ops/)

## References

1. L. Schmidt, D. Van der Spoel & M.-M. Walz, *ACS Phys. Chem. Au* **3**, 84–93 (2023). DOI: [10.1021/acsphyschemau.2c00045](https://doi.org/10.1021/acsphyschemau.2c00045) — solid–liquid coexistence with the rotational order parameter $S_0$ and diffusion coefficient $D$ as the melting-point screen.
2. C. P. Brock & J. D. Dunitz, *Acta Crystallogr. B* **38**, 2218–2228 (1982). DOI: [10.1107/S0567740882008358](https://doi.org/10.1107/S0567740882008358) — naphthalene crystal structure (CSD NAPHTA10, CCDC 1216816).
3. B. Rhodes et al., *Orb-v3: atomistic simulation at scale*, [arXiv:2504.06231](https://arxiv.org/abs/2504.06231) (2025) — the Orb-v3 (OMol) foundation potential.
4. L. Martínez, R. Andrade, E. G. Birgin & J. M. Martínez, *J. Comput. Chem.* **30**, 2157–2164 (2009). DOI: [10.1002/jcc.21224](https://doi.org/10.1002/jcc.21224) — Packmol.
5. NIST Chemistry WebBook, *Naphthalene* (CAS 91-20-3), fusion temperature $T_\mathrm{fus} = 353.2 \pm 0.7\ \mathrm{K}$. NIST Standard Reference Database 69. [webbook.nist.gov](https://webbook.nist.gov/cgi/cbook.cgi?ID=C91203) — the experimental melting point.